# Stratified Training Set A/B Validation

**Intent:** Test whether matching training movies on `(first_review → close)` gap to the target movie's gap reduces lambda MAE vs the current recency-only training set (`default_training_slugs`, n=20 most recent).

**Hypothesis:** Convolution identity says `Var(close-anchor timing) = Var(embargo-anchor timing) + Var(gap_training)`. By selecting training movies with similar gaps to the target, we shrink `Var(gap_training)` toward zero — getting embargo-anchor's conditioning benefit *without* needing real embargo data. Should help most on long-tail (Q3/Q4 gap) targets where the current model extrapolates outside training support.

**Methodology:** Same Step 4 LOO from `findings/embargo_anchor_investigation.md` §4, swapped intervention. For each resolved movie, build baseline + stratified profiles, snapshot at T-3d and T-1d, compare predicted vs actual remaining reviews.

**Decision rule:** Stratified beats baseline by ≥10% MAE overall OR ≥20% on Q3/Q4 alone → green light to consider library changes.

**Caching:** LOO results stored in `notebooks/.cache/stratified_training_loo.parquet` keyed by (target_slug, snapshot_dbc, method). Method names include the band (e.g., `stratified_band_3`). Re-running cells skips cached entries.

In [ ]:
import sys
from pathlib import Path

# Workaround for PEP 660 editable install not loading in jupyter kernel subprocess
# (see findings/embargo_anchor_investigation.md \u00a78)
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rotten_tomatoes_forecasting import (
    build_critic_profiles, build_kde_lambda_model,
    estimate_lambda, default_training_slugs,
)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
})

CACHE_DIR = ROOT / 'notebooks' / '.cache'
CACHE_DIR.mkdir(exist_ok=True)
CACHE_PATH = CACHE_DIR / 'stratified_training_loo.pkl'

# -- Load and parse ---------------------------------------------------------
reviews = pd.read_csv(ROOT / 'reviews.csv')
reviews['estimated_timestamp'] = pd.to_datetime(
    reviews['estimated_timestamp'], format='ISO8601', utc=True,
)

movies = pd.read_csv(ROOT / 'movies_index.csv')
movies['Bet Close Date'] = pd.to_datetime(
    movies['Bet Close Date'], utc=True, errors='coerce',
)

# Resolved = movies with a Bet Close Date in the past
now = pd.Timestamp.now(tz='UTC')
resolved = movies.dropna(subset=['Bet Close Date'])
resolved = resolved[resolved['Bet Close Date'] < now]
print(f'Resolved movies: {len(resolved)}')

close_date_map = resolved.set_index('Slug')['Bet Close Date'].to_dict()

first_review_ts = (
    reviews[reviews['movie_slug'].isin(close_date_map)]
    .groupby('movie_slug')['estimated_timestamp'].min()
)
print(f'Movies with at least one review: {first_review_ts.notna().sum()}')

assert reviews['estimated_timestamp'].notna().all(), 'NaT timestamps in reviews'
assert reviews['estimated_timestamp'].dt.tz is not None, 'Timestamps must be tz-aware'

## 1. Cohort gap diagnostics

Compute `(close − first_review)` per movie. Establishes the cohort distribution and the quantile cutoffs we'll use to stratify A/B results. Reproduces Step 1 from `findings/embargo_anchor_investigation.md`.

In [ ]:
gaps = (
    first_review_ts
    .rename('first_review_ts')
    .reset_index()
    .rename(columns={'movie_slug': 'slug'})
)
gaps['close_ts'] = gaps['slug'].map(close_date_map)
gaps['gap_days'] = (gaps['close_ts'] - gaps['first_review_ts']).dt.total_seconds() / 86400
gaps = gaps.dropna(subset=['close_ts', 'gap_days'])
gaps = gaps[gaps['gap_days'] > 0].reset_index(drop=True)

print(f'Cohort size with valid gaps: {len(gaps)}')
print()
print('Gap distribution (days from first review to close):')
print(gaps['gap_days'].describe([.25, .5, .75, .9]).round(2).to_string())

q_cutoffs = gaps['gap_days'].quantile([0.25, 0.5, 0.75]).values
print()
print(f'Quantile cutoffs: Q1\u2264{q_cutoffs[0]:.2f}d, Q2\u2264{q_cutoffs[1]:.2f}d, Q3\u2264{q_cutoffs[2]:.2f}d')

fig, ax = plt.subplots()
ax.hist(gaps['gap_days'].clip(upper=60), bins=40, color='steelblue', alpha=0.7)
for c, label in zip(q_cutoffs, ['Q1', 'Q2', 'Q3']):
    ax.axvline(c, color='red', ls='--', alpha=0.5, label=f'{label}={c:.1f}d')
ax.set_xlabel('Gap (close \u2212 first_review), days, clipped at 60')
ax.set_ylabel('Count')
ax.set_title('Cohort gap distribution')
ax.legend()
plt.tight_layout()

## 2. Matched training-set selector

`matched_training_slugs(target, target_gap, band, n=20)`: pick the `n` most recent movies before `target.close` whose gap is within `±band` of `target_gap`. If <`n` matches, expand the band in 0.5d steps until `n` found. Excludes target.

Spot-check vs baseline below should show stratified training has tighter gap distribution.

In [ ]:
gap_lookup = dict(zip(gaps['slug'], gaps['gap_days']))

def gap_for_slug(slug):
    return gap_lookup.get(slug)

def matched_training_slugs(target_slug, target_gap, band, n=20):
    """Return (slugs, effective_band). Expands band until n found."""
    target_close = close_date_map[target_slug]
    candidates = gaps[
        (gaps['close_ts'] < target_close)
        & (gaps['slug'] != target_slug)
    ].copy()

    current = band
    while True:
        matched = candidates[(candidates['gap_days'] - target_gap).abs() <= current]
        if len(matched) >= n or current > 1000:
            break
        current += 0.5
    selected = matched.sort_values('close_ts', ascending=False).head(n)
    return selected['slug'].tolist(), current

# Spot-check on a recent movie
sample_target = gaps.sort_values('close_ts').iloc[-30]['slug']
sample_gap = gap_for_slug(sample_target)
print(f'Sample target: {sample_target} (gap = {sample_gap:.2f}d)')

baseline = default_training_slugs(
    movies, exclude_slug=sample_target,
    before_date=close_date_map[sample_target], n=20,
)
stratified, eff_band = matched_training_slugs(sample_target, sample_gap, band=3.0, n=20)

print()
print(f'Baseline ({len(baseline)}):    first 5 = {baseline[:5]}')
print(f'Stratified ({len(stratified)}, band={eff_band:.1f}d): first 5 = {stratified[:5]}')

baseline_gaps = [gap_for_slug(s) for s in baseline if gap_for_slug(s) is not None]
stratified_gaps = [gap_for_slug(s) for s in stratified if gap_for_slug(s) is not None]
print()
print(f'Baseline training gaps:   median={np.median(baseline_gaps):.2f}d, σ={np.std(baseline_gaps):.2f}d')
print(f'Stratified training gaps: median={np.median(stratified_gaps):.2f}d, σ={np.std(stratified_gaps):.2f}d')

## 3. LOO A/B loop with caching

Per target: build baseline + stratified profiles+KDE *once each* (snapshots reuse the same model). At each snapshot:

- `observed_state` from reviews with `estimated_timestamp < snap_time`
- `predicted_remaining = estimate_lambda × hours_to_close`
- `actual_remaining = reviews with 0 < dbc ≤ snap_dbc` (matches library's internal filter; symmetric across methods)

Cache is keyed by (target_slug, snapshot_dbc, method) where method = `baseline` or `stratified_band_X`. Re-running for a different band only computes the new method.

In [ ]:
SNAPSHOTS = [3.0, 1.0]  # T-3d, T-1d

def snapshot_state(target_slug, snap_time):
    target_close = close_date_map[target_slug]
    obs = reviews[
        (reviews['movie_slug'] == target_slug)
        & (reviews['estimated_timestamp'] < snap_time)
        & (reviews['estimated_timestamp'] < target_close)
    ]
    if obs.empty:
        return None
    return {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float(
            (target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400
        ),
    }

def actual_remaining(target_slug, snap_dbc):
    target_close = close_date_map[target_slug]
    movie_reviews = reviews[reviews['movie_slug'] == target_slug].copy()
    movie_reviews['dbc'] = (
        target_close - movie_reviews['estimated_timestamp']
    ).dt.total_seconds() / 86400
    return int(((movie_reviews['dbc'] > 0) & (movie_reviews['dbc'] <= snap_dbc)).sum())

In [ ]:
def run_loo(band, force=False, verbose_progress=True):
    """Run LOO for one band. Returns the full (combined) cache DataFrame.

    Skips (target, method) pairs already in cache unless force=True. Baseline only
    needs to be computed once across band sweeps.
    """
    method_key = f'stratified_band_{band:g}'
    cached = pd.read_pickle(CACHE_PATH) if CACHE_PATH.exists() else pd.DataFrame()

    targets = list(close_date_map.keys())
    new_rows = []
    for i, target in enumerate(targets):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue

        for method in ['baseline', method_key]:
            # Skip if both snapshots already cached
            if not force and not cached.empty:
                hit = cached[
                    (cached['target_slug'] == target)
                    & (cached['method'] == method)
                ]
                if len(hit) >= len(SNAPSHOTS):
                    continue

            # Build training set
            if method == 'baseline':
                training = default_training_slugs(
                    movies, exclude_slug=target,
                    before_date=close_date_map[target], n=20,
                )
            else:
                training, _ = matched_training_slugs(target, target_gap, band=band, n=20)

            if len(training) < 5:
                for snap_dbc in SNAPSHOTS:
                    new_rows.append({
                        'target_slug': target, 'target_gap': target_gap,
                        'snapshot_dbc': snap_dbc, 'method': method,
                        'predicted': np.nan, 'actual': np.nan,
                    })
                continue

            # Build profiles + KDE ONCE per (target, method)
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model(profiles, verbose=False)

            target_close = close_date_map[target]
            for snap_dbc in SNAPSHOTS:
                snap_time = target_close - pd.Timedelta(days=snap_dbc)
                state = snapshot_state(target, snap_time)
                if state is None:
                    new_rows.append({
                        'target_slug': target, 'target_gap': target_gap,
                        'snapshot_dbc': snap_dbc, 'method': method,
                        'predicted': np.nan, 'actual': np.nan,
                    })
                    continue

                htc = snap_dbc * 24
                lam = estimate_lambda(
                    model, snap_dbc, htc,
                    observed_critics=state['observed_critics'],
                    observed_count=state['observed_count'],
                    first_review_dbc=state['first_review_dbc'],
                )
                new_rows.append({
                    'target_slug': target, 'target_gap': target_gap,
                    'snapshot_dbc': snap_dbc, 'method': method,
                    'predicted': float(lam * htc),
                    'actual': actual_remaining(target, snap_dbc),
                })

        if verbose_progress and (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(targets)} targets processed')

    new = pd.DataFrame(new_rows)
    if not new.empty and not cached.empty:
        combined = pd.concat([cached, new], ignore_index=True)
    elif not new.empty:
        combined = new
    else:
        combined = cached
    if not combined.empty:
        combined = combined.drop_duplicates(
            ['target_slug', 'snapshot_dbc', 'method'], keep='last',
        )
        combined.to_pickle(CACHE_PATH)
        print(f'Cached {len(combined)} total rows ({len(new)} new) to {CACHE_PATH.name}')
    return combined

In [ ]:
results = run_loo(band=3.0)
print()
print(results.head(8).to_string(index=False))

## 4. Overall A/B comparison

MAE / median_err / p90|err| at T-3d and T-1d, baseline vs stratified. Restricted to targets with both methods present (apples-to-apples).

In [ ]:
def ab_summary(results, method_a='baseline', method_b='stratified_band_3'):
    df = results.dropna(subset=['predicted', 'actual']).copy()
    df = df[df['method'].isin([method_a, method_b])]
    df['err'] = df['predicted'] - df['actual']
    df['abs_err'] = df['err'].abs()

    out_rows = []
    rel_changes = []
    for snap in sorted(df['snapshot_dbc'].unique(), reverse=True):
        snap_df = df[df['snapshot_dbc'] == snap]
        common = snap_df.groupby('target_slug')['method'].nunique() == 2
        snap_df = snap_df[snap_df['target_slug'].isin(common[common].index)]

        per_method = {}
        for method in [method_a, method_b]:
            m = snap_df[snap_df['method'] == method]
            per_method[method] = {
                'MAE': m['abs_err'].mean(),
                'median_err': m['err'].median(),
                'p90_abs_err': m['abs_err'].quantile(0.9),
                'n': len(m),
            }
            out_rows.append({'snapshot': f'T-{snap:g}d', 'method': method, **per_method[method]})

        a_mae = per_method[method_a]['MAE']
        b_mae = per_method[method_b]['MAE']
        rel = (a_mae - b_mae) / a_mae * 100 if a_mae else 0
        rel_changes.append((snap, rel))

    summary = pd.DataFrame(out_rows)[['snapshot', 'method', 'n', 'MAE', 'median_err', 'p90_abs_err']]
    print(summary.to_string(index=False, float_format='%.2f'))
    print()
    for snap, rel in rel_changes:
        sign = 'BETTER' if rel > 0 else 'WORSE'
        print(f'T-{snap:g}d MAE: {method_b} vs {method_a} = {rel:+.1f}% ({sign})')

ab_summary(results, method_a='baseline', method_b='stratified_band_3')

## 5. A/B by target gap quantile

Stratified-training should help most where the baseline training set's gap distribution is most unrepresentative — i.e., on Q3/Q4 (long-gap) targets.

In [ ]:
def ab_by_quantile(results, method_a='baseline', method_b='stratified_band_3'):
    df = results.dropna(subset=['predicted', 'actual']).copy()
    df = df[df['method'].isin([method_a, method_b])]
    df['err'] = df['predicted'] - df['actual']
    df['abs_err'] = df['err'].abs()

    def quantile_bin(g):
        if g <= q_cutoffs[0]:
            return 'Q1'
        if g <= q_cutoffs[1]:
            return 'Q2'
        if g <= q_cutoffs[2]:
            return 'Q3'
        return 'Q4'
    df['quantile'] = df['target_gap'].apply(quantile_bin)

    for snap in sorted(df['snapshot_dbc'].unique(), reverse=True):
        snap_df = df[df['snapshot_dbc'] == snap]
        common = snap_df.groupby('target_slug')['method'].nunique() == 2
        snap_df = snap_df[snap_df['target_slug'].isin(common[common].index)]

        print(f'\n=== T-{snap:g}d MAE by gap quantile ===')
        rows = []
        for q in ['Q1', 'Q2', 'Q3', 'Q4']:
            qdf = snap_df[snap_df['quantile'] == q]
            a_mae = qdf[qdf['method'] == method_a]['abs_err'].mean()
            b_mae = qdf[qdf['method'] == method_b]['abs_err'].mean()
            rel = (a_mae - b_mae) / a_mae * 100 if a_mae else 0
            rows.append({
                'quantile': q,
                'n': qdf['target_slug'].nunique(),
                f'{method_a}_MAE': a_mae,
                f'{method_b}_MAE': b_mae,
                'rel_change_pct': rel,
            })
        print(pd.DataFrame(rows).to_string(index=False, float_format='%.2f'))

ab_by_quantile(results)

## 6. Band sweep

Run additional bands and compare. Tighter bands should give more variance reduction but smaller per-target training samples (more band-expansion).

In [ ]:
results = run_loo(band=2.0)
results = run_loo(band=5.0)

for band in [2.0, 3.0, 5.0]:
    method_b = f'stratified_band_{band:g}'
    print(f'\n{"="*60}\nBAND = {band:g}d\n{"="*60}')
    ab_summary(results, method_b=method_b)
    ab_by_quantile(results, method_b=method_b)

## 7. Sanity checks

- Sample sizes per (snapshot, method) should be similar within snapshot (no method-specific dropouts).
- Spot-check 5 random targets: stratified training median gap should be closer to target's gap than baseline's.
- Band-expansion frequency: how often does the band have to grow to find 20 matches?

In [ ]:
print('Sample sizes per (snapshot, method):')
print(
    results.dropna(subset=['predicted'])
    .groupby(['snapshot_dbc', 'method']).size().unstack(fill_value=0)
)

print('\nSpot-check (target_gap vs baseline median gap vs stratified median gap):')
rng = np.random.default_rng(42)
sample_targets = rng.choice(gaps['slug'].values, size=5, replace=False)
for t in sample_targets:
    g = gap_for_slug(t)
    baseline_t = default_training_slugs(
        movies, exclude_slug=t, before_date=close_date_map[t], n=20,
    )
    stratified_t, eff_band = matched_training_slugs(t, g, band=3.0, n=20)
    base_med = np.median([gap_for_slug(s) for s in baseline_t if gap_for_slug(s) is not None])
    strat_med = np.median([gap_for_slug(s) for s in stratified_t if gap_for_slug(s) is not None])
    print(f'  {t}: target={g:.1f}d, baseline_med={base_med:.1f}d, '
          f'stratified_med={strat_med:.1f}d (band→{eff_band:.1f}d)')

print('\nBand-expansion frequency at band=3.0d (over all targets):')
expansions = []
for t in gaps['slug']:
    g = gap_for_slug(t)
    _, eff = matched_training_slugs(t, g, band=3.0, n=20)
    expansions.append(eff)
exp_series = pd.Series(expansions)
print(f'  effective band: median={exp_series.median():.1f}d, '
      f'p75={exp_series.quantile(0.75):.1f}d, '
      f'p90={exp_series.quantile(0.9):.1f}d, max={exp_series.max():.1f}d')
print(f'  fraction expanded beyond 3d: {(exp_series > 3.0).mean():.1%}')

## 8. Close-day piecewise patch

Tests whether adding an aggregate close-day window estimate closes the T-1d regression observed in stratified training (\u00a76.2). Design: 2x2 factorial of {baseline, stratified} \u00d7 {no-piecewise, +piecewise}.

**Mechanics:**
- `F` = fraction of close-day reviews that arrive in 12am UTC \u2013 10am EST window. Estimated from `the_drama` and `the_super_mario_galaxy_movie` (only movies with usable h/m close-day data).
- **Phase 1 prediction** = existing model output (predicts to midnight UTC of close day).
- **Phase 2 prediction** = `F \u00d7 mean_close_day_count_in_training` (added on top for piecewise methods).
- **Expanded actual** = existing `actual_remaining` + `F \u00d7 close_day_count(target)`. Applied to ALL four methods so the comparison is apples-to-apples.

**Decision criterion:** Piecewise variants close the T-1d regression (negative \u2192 zero or positive). Stretch goal: stratified+piecewise wins overall at both T-3d and T-1d.

In [ ]:
EARLY_MOVIES = ['the_drama', 'the_super_mario_galaxy_movie']
PRE_MARKET_HOURS = 14  # midnight UTC \u2192 10am EST = 14:00 UTC (using EDT; \u00b11h depending on DST)

def estimate_f(slug):
    target_close = close_date_map[slug]
    market_close_time = target_close + pd.Timedelta(hours=PRE_MARKET_HOURS)
    movie_reviews = reviews[
        (reviews['movie_slug'] == slug)
        & (reviews['timestamp_confidence'].isin(['m', 'h']))
        & (reviews['estimated_timestamp'].dt.floor('D') == target_close.floor('D'))
    ]
    if len(movie_reviews) == 0:
        return None, 0
    pre_market = (movie_reviews['estimated_timestamp'] < market_close_time).sum()
    return pre_market / len(movie_reviews), len(movie_reviews)

print('F estimation:')
f_results = {}
for slug in EARLY_MOVIES:
    f, n = estimate_f(slug)
    f_results[slug] = (f, n)
    print(f'  {slug}: F = {f:.3f} (n={n} h/m close-day reviews)')

valid_fs = [v[0] for v in f_results.values() if v[0] is not None]
F = float(np.mean(valid_fs))
print(f'\nAggregate F (mean across movies) = {F:.3f}')

def close_day_count(slug):
    """Count reviews timestamped to the close day (any confidence)."""
    target_close = close_date_map[slug]
    movie_reviews = reviews[reviews['movie_slug'] == slug]
    same_day = movie_reviews['estimated_timestamp'].dt.floor('D') == target_close.floor('D')
    return int(same_day.sum())

# Sanity check: close-day counts for a few targets
print('\nClose-day review counts (sample):')
sample_slugs = list(close_date_map.keys())[:10]
for s in sample_slugs:
    print(f'  {s}: {close_day_count(s)}')

In [ ]:
def run_loo_piecewise(band, force=False, verbose_progress=True):
    """Compute piecewise predictions for baseline + stratified at given band.

    Reuses run_loo's training-set logic but adds Phase 2 = F * mean_close_day_count
    over training movies. Cache key uses '_piecewise' suffix.
    """
    cached = pd.read_pickle(CACHE_PATH) if CACHE_PATH.exists() else pd.DataFrame()
    targets = list(close_date_map.keys())
    new_rows = []

    method_specs = [
        ('baseline_piecewise', lambda t, g: default_training_slugs(
            movies, exclude_slug=t, before_date=close_date_map[t], n=20,
        )),
        (f'stratified_band_{band:g}_piecewise', lambda t, g: matched_training_slugs(
            t, g, band=band, n=20,
        )[0]),
    ]

    for i, target in enumerate(targets):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue

        for method_name, training_fn in method_specs:
            # Skip if both snapshots cached
            if not force and not cached.empty:
                hit = cached[
                    (cached['target_slug'] == target)
                    & (cached['method'] == method_name)
                ]
                if len(hit) >= len(SNAPSHOTS):
                    continue

            training = training_fn(target, target_gap)
            if len(training) < 5:
                for snap_dbc in SNAPSHOTS:
                    new_rows.append({
                        'target_slug': target, 'target_gap': target_gap,
                        'snapshot_dbc': snap_dbc, 'method': method_name,
                        'predicted': np.nan, 'actual': np.nan,
                    })
                continue

            # Phase 2: F * mean close-day count across training movies
            mean_close_day = float(np.mean([close_day_count(s) for s in training]))
            phase2 = F * mean_close_day

            # Phase 1: build profiles + KDE, predict per snapshot
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model(profiles, verbose=False)

            target_close = close_date_map[target]
            for snap_dbc in SNAPSHOTS:
                snap_time = target_close - pd.Timedelta(days=snap_dbc)
                state = snapshot_state(target, snap_time)
                if state is None:
                    new_rows.append({
                        'target_slug': target, 'target_gap': target_gap,
                        'snapshot_dbc': snap_dbc, 'method': method_name,
                        'predicted': np.nan, 'actual': np.nan,
                    })
                    continue

                htc = snap_dbc * 24
                lam = estimate_lambda(
                    model, snap_dbc, htc,
                    observed_critics=state['observed_critics'],
                    observed_count=state['observed_count'],
                    first_review_dbc=state['first_review_dbc'],
                )
                phase1 = float(lam * htc)
                new_rows.append({
                    'target_slug': target, 'target_gap': target_gap,
                    'snapshot_dbc': snap_dbc, 'method': method_name,
                    'predicted': phase1 + phase2,
                    'actual': actual_remaining(target, snap_dbc),  # un-expanded; expanded in summary
                })

        if verbose_progress and (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(targets)} targets processed')

    new = pd.DataFrame(new_rows)
    if not new.empty and not cached.empty:
        combined = pd.concat([cached, new], ignore_index=True)
    elif not new.empty:
        combined = new
    else:
        combined = cached
    if not combined.empty:
        combined = combined.drop_duplicates(['target_slug', 'snapshot_dbc', 'method'], keep='last')
        combined.to_pickle(CACHE_PATH)
        print(f'Cached {len(combined)} total rows ({len(new)} new)')
    return combined

results = run_loo_piecewise(band=3.0)

In [ ]:
METHODS_4 = [
    'baseline',
    'stratified_band_3',
    'baseline_piecewise',
    'stratified_band_3_piecewise',
]

def expand_actuals(df):
    """Add F * close_day_count(target) to stored actual. Symmetric across all methods."""
    df = df.copy()
    cd_lookup = {s: close_day_count(s) for s in df['target_slug'].unique()}
    df['actual_expanded'] = df['actual'] + F * df['target_slug'].map(cd_lookup)
    df['err'] = df['predicted'] - df['actual_expanded']
    df['abs_err'] = df['err'].abs()
    return df

def ab_summary_4(results, methods=METHODS_4):
    df = expand_actuals(results.dropna(subset=['predicted', 'actual']))
    df = df[df['method'].isin(methods)]

    out_rows = []
    for snap in sorted(df['snapshot_dbc'].unique(), reverse=True):
        snap_df = df[df['snapshot_dbc'] == snap]
        common = snap_df.groupby('target_slug')['method'].nunique() == len(methods)
        snap_df = snap_df[snap_df['target_slug'].isin(common[common].index)]

        for method in methods:
            m = snap_df[snap_df['method'] == method]
            out_rows.append({
                'snapshot': f'T-{snap:g}d',
                'method': method,
                'n': len(m),
                'MAE': m['abs_err'].mean(),
                'median_err': m['err'].median(),
                'p90_abs_err': m['abs_err'].quantile(0.9),
            })

    summary = pd.DataFrame(out_rows)
    print(summary.to_string(index=False, float_format='%.2f'))

    print('\n--- MAE deltas (vs baseline) ---')
    for snap in sorted(df['snapshot_dbc'].unique(), reverse=True):
        baseline_mae = summary[
            (summary['snapshot'] == f'T-{snap:g}d') & (summary['method'] == 'baseline')
        ]['MAE'].iloc[0]
        for method in methods[1:]:
            mae = summary[
                (summary['snapshot'] == f'T-{snap:g}d') & (summary['method'] == method)
            ]['MAE'].iloc[0]
            rel = (baseline_mae - mae) / baseline_mae * 100
            sign = 'BETTER' if rel > 0 else 'WORSE'
            print(f'  T-{snap:g}d  {method:35s}  {rel:+6.1f}%  ({sign})')

ab_summary_4(results)

In [ ]:
def ab_by_quantile_4(results, methods=METHODS_4):
    df = expand_actuals(results.dropna(subset=['predicted', 'actual']))
    df = df[df['method'].isin(methods)]

    def quantile_bin(g):
        if g <= q_cutoffs[0]: return 'Q1'
        if g <= q_cutoffs[1]: return 'Q2'
        if g <= q_cutoffs[2]: return 'Q3'
        return 'Q4'
    df['quantile'] = df['target_gap'].apply(quantile_bin)

    for snap in sorted(df['snapshot_dbc'].unique(), reverse=True):
        snap_df = df[df['snapshot_dbc'] == snap]
        common = snap_df.groupby('target_slug')['method'].nunique() == len(methods)
        snap_df = snap_df[snap_df['target_slug'].isin(common[common].index)]

        print(f'\n=== T-{snap:g}d MAE by gap quantile ===')
        rows = []
        for q in ['Q1', 'Q2', 'Q3', 'Q4']:
            qdf = snap_df[snap_df['quantile'] == q]
            row = {'quantile': q, 'n': qdf['target_slug'].nunique()}
            for method in methods:
                # Shorten method name for display
                short = method.replace('stratified_band_3', 'strat').replace('_piecewise', '+pw')
                row[short] = qdf[qdf['method'] == method]['abs_err'].mean()
            rows.append(row)
        print(pd.DataFrame(rows).to_string(index=False, float_format='%.2f'))

ab_by_quantile_4(results)

### 8.1 Cohort-wide h/m close-day audit

Per Jake's note: only a handful of movies were live-scraped or backfilled near close. Quick scan to confirm we're not missing meaningful additional h/m close-day samples that could tighten the F estimate.

Also flagged: for `the_drama` and `the_super_mario_galaxy_movie` specifically, the reviews.csv pull was right around 11am EST on bet-close day — so their h/m close-day reviews are essentially pre-market arrivals captured before the post-market window opened. F=1 from these two reflects this capture window as much as critic behavior.

In [ ]:
hm_cd_rows = []
for slug in close_date_map:
    target_close = close_date_map[slug]
    movie_reviews = reviews[reviews['movie_slug'] == slug]
    hm_cd = movie_reviews[
        (movie_reviews['timestamp_confidence'].isin(['m', 'h']))
        & (movie_reviews['estimated_timestamp'].dt.floor('D') == target_close.floor('D'))
    ]
    if len(hm_cd) > 0:
        pre = (hm_cd['estimated_timestamp'] < target_close + pd.Timedelta(hours=PRE_MARKET_HOURS)).sum()
        hm_cd_rows.append({
            'slug': slug,
            'hm_close_day_count': len(hm_cd),
            'pre_market': int(pre),
            'post_market': len(hm_cd) - int(pre),
        })

hm_df = pd.DataFrame(hm_cd_rows).sort_values('hm_close_day_count', ascending=False)
print(f'Movies with at least one h/m close-day review: {len(hm_df)}/{len(close_date_map)}')
print(f'Total h/m close-day reviews across cohort: {hm_df["hm_close_day_count"].sum()}')
print(f'  Pre-market (before {PRE_MARKET_HOURS}h after midnight UTC): {hm_df["pre_market"].sum()}')
print(f'  Post-market: {hm_df["post_market"].sum()}')
print()
print(hm_df.head(20).to_string(index=False))

### 8.2 F sensitivity

Cached predictions and actuals were built with F=1.0. To test sensitivity at F ∈ {0.5, 0.7, 1.0} without re-running the LOO loop, decompose:

- **Piecewise predicted at F_test** = `cached_predicted − F_baked × mean_cd_training + F_test × mean_cd_training`
- **Expanded actual at F_test** = `cached_actual + F_test × close_day_count(target)`

Both adjustments use the same F_test value, applied symmetrically. Decision: piecewise's T-1d win is reliable if it survives across the F range.

In [ ]:
F_BAKED_IN = 1.0  # F value used when caching predictions

# Pre-compute mean_cd_training per (target, method) — needed to decompose cached predictions
print('Pre-computing mean_cd_training per (target, method)...')
mean_cd_cache = {}
for target in close_date_map:
    target_gap = gap_for_slug(target)
    if target_gap is None:
        continue
    base_training = default_training_slugs(
        movies, exclude_slug=target, before_date=close_date_map[target], n=20,
    )
    mean_cd_cache[(target, 'baseline_piecewise')] = float(
        np.mean([close_day_count(s) for s in base_training])
    )
    strat_training, _ = matched_training_slugs(target, target_gap, band=3.0, n=20)
    mean_cd_cache[(target, 'stratified_band_3_piecewise')] = float(
        np.mean([close_day_count(s) for s in strat_training])
    )

target_cd_cache = {s: close_day_count(s) for s in close_date_map}
print('Done.')

def sensitivity_run(F_test):
    df = results.dropna(subset=['predicted', 'actual']).copy()
    df = df[df['method'].isin(METHODS_4)]

    # Adjust predictions: piecewise methods get phase2 swapped from F_BAKED_IN to F_test
    def adj_pred(row):
        if 'piecewise' not in row['method']:
            return row['predicted']
        mean_cd = mean_cd_cache.get((row['target_slug'], row['method']), 0.0)
        return row['predicted'] - F_BAKED_IN * mean_cd + F_test * mean_cd
    df['predicted_adj'] = df.apply(adj_pred, axis=1)

    # Adjust actuals: all methods get F_test * close_day_count(target) added
    df['actual_adj'] = df['actual'] + F_test * df['target_slug'].map(target_cd_cache)
    df['err'] = df['predicted_adj'] - df['actual_adj']
    df['abs_err'] = df['err'].abs()

    out_rows = []
    for snap in sorted(df['snapshot_dbc'].unique(), reverse=True):
        snap_df = df[df['snapshot_dbc'] == snap]
        common = snap_df.groupby('target_slug')['method'].nunique() == len(METHODS_4)
        snap_df = snap_df[snap_df['target_slug'].isin(common[common].index)]
        baseline_mae = snap_df[snap_df['method'] == 'baseline']['abs_err'].mean()
        for method in METHODS_4:
            mae = snap_df[snap_df['method'] == method]['abs_err'].mean()
            med_err = snap_df[snap_df['method'] == method]['err'].median()
            rel = (baseline_mae - mae) / baseline_mae * 100 if baseline_mae else 0
            out_rows.append({
                'snapshot': f'T-{snap:g}d',
                'method': method.replace('stratified_band_3', 'strat').replace('_piecewise', '+pw'),
                'MAE': mae,
                'median_err': med_err,
                'rel_vs_baseline': f'{rel:+.1f}%',
            })
    return pd.DataFrame(out_rows)

for F_test in [0.5, 0.7, 1.0]:
    print(f'\n{"="*55}\nF_test = {F_test}\n{"="*55}')
    print(sensitivity_run(F_test).to_string(index=False, float_format='%.2f'))

### 8.3 F sensitivity on day-level-only subset

The 4 movies with h/m close-day reviews have ambiguous `close_day_count` interpretation: their counts may already be ~pre-market arrivals (if reviews.csv was pulled mid-close-day) rather than full close-day totals. This means applying `F × close_day_count` could double-discount or overcount on these targets specifically.

Cleanest fix: exclude them from validation targets entirely. We can no longer derive F from these movies (they're out), so just sweep F as a parameter and find the value that minimizes MAE on day-level-only targets.

This converts F from "estimated parameter" to "tunable hyperparameter" — and the optimal F under this evaluation is our best deployment recommendation.

In [ ]:
EXCLUDE_SLUGS = {
    'the_drama',
    'the_super_mario_galaxy_movie',
    'they_will_kill_you',
    'forbidden_fruits_2026',
}

def sensitivity_run_excl(F_test, exclude=EXCLUDE_SLUGS):
    df = results.dropna(subset=['predicted', 'actual']).copy()
    df = df[df['method'].isin(METHODS_4)]
    df = df[~df['target_slug'].isin(exclude)]

    def adj_pred(row):
        if 'piecewise' not in row['method']:
            return row['predicted']
        mean_cd = mean_cd_cache.get((row['target_slug'], row['method']), 0.0)
        return row['predicted'] - F_BAKED_IN * mean_cd + F_test * mean_cd
    df['predicted_adj'] = df.apply(adj_pred, axis=1)

    df['actual_adj'] = df['actual'] + F_test * df['target_slug'].map(target_cd_cache)
    df['err'] = df['predicted_adj'] - df['actual_adj']
    df['abs_err'] = df['err'].abs()

    out_rows = []
    for snap in sorted(df['snapshot_dbc'].unique(), reverse=True):
        snap_df = df[df['snapshot_dbc'] == snap]
        common = snap_df.groupby('target_slug')['method'].nunique() == len(METHODS_4)
        snap_df = snap_df[snap_df['target_slug'].isin(common[common].index)]
        baseline_mae = snap_df[snap_df['method'] == 'baseline']['abs_err'].mean()
        for method in METHODS_4:
            mae = snap_df[snap_df['method'] == method]['abs_err'].mean()
            med_err = snap_df[snap_df['method'] == method]['err'].median()
            rel = (baseline_mae - mae) / baseline_mae * 100 if baseline_mae else 0
            out_rows.append({
                'snapshot': f'T-{snap:g}d',
                'method': method.replace('stratified_band_3', 'strat').replace('_piecewise', '+pw'),
                'MAE': mae,
                'median_err': med_err,
                'rel_vs_baseline': f'{rel:+.1f}%',
            })
    return pd.DataFrame(out_rows)

print(f'Excluding {len(EXCLUDE_SLUGS)} live-tracked movies with h/m close-day reviews')
remaining = (
    results.dropna(subset=['predicted', 'actual'])
    [~results['target_slug'].isin(EXCLUDE_SLUGS)]
    ['target_slug'].nunique()
)
print(f'Validation targets remaining: {remaining}')

for F_test in [0.0, 0.3, 0.5, 0.7, 1.0]:
    print(f'\n{"="*55}\nF_test = {F_test}\n{"="*55}')
    print(sensitivity_run_excl(F_test).to_string(index=False, float_format='%.2f'))

In [ ]:
# Finer-grained F sweep to find the optimum, day-level-only
F_grid = np.arange(0.0, 1.05, 0.1)
optim_rows = []
for F_test in F_grid:
    summary = sensitivity_run_excl(F_test)
    for snap in ['T-3d', 'T-1d']:
        for method in ['baseline', 'strat', 'baseline+pw', 'strat+pw']:
            mae = summary[(summary['snapshot'] == snap) & (summary['method'] == method)]['MAE'].iloc[0]
            optim_rows.append({'F': F_test, 'snapshot': snap, 'method': method, 'MAE': mae})

opt_df = pd.DataFrame(optim_rows)
pivot = opt_df.pivot_table(index='F', columns=['snapshot', 'method'], values='MAE')
print('MAE vs F (day-level-only subset, n={})'.format(remaining))
print(pivot.round(2).to_string())

print('\nOptimal F per method per snapshot:')
for snap in ['T-3d', 'T-1d']:
    for method in ['baseline+pw', 'strat+pw']:
        sub = opt_df[(opt_df['snapshot'] == snap) & (opt_df['method'] == method)]
        best = sub.loc[sub['MAE'].idxmin()]
        print(f'  {snap} {method:12s}: F*={best["F"]:.1f}  MAE={best["MAE"]:.2f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, snap in zip(axes, ['T-3d', 'T-1d']):
    for method in ['baseline', 'strat', 'baseline+pw', 'strat+pw']:
        sub = opt_df[(opt_df['snapshot'] == snap) & (opt_df['method'] == method)]
        ax.plot(sub['F'], sub['MAE'], marker='o', label=method)
    ax.set_xlabel('F')
    ax.set_ylabel('MAE')
    ax.set_title(f'{snap} (day-level subset)')
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()

## 9. KDE quality test (close-day-free middle window)

Pure test of KDE prediction quality, isolated from close-day handling. For each target, snap at T-3d and predict reviews in the **(T-3d, T-1d]** window (2-day middle slice). The window doesn't touch close day, so close-day mass is not in scope for either predicted or actual.

If KDE is well-calibrated here (median_err ≈ 0), then the close-day handling is the localized issue and the rest of the model is solid. If there's residual bias, the KDE itself has issues that piecewise won't fix.

Reports both full cohort and day-level-only subset for comparability.


In [ ]:
from rotten_tomatoes_forecasting.critic_model import _blended_integral, _compute_scaling

def predict_window(model, dbc_from, dbc_to, observed_critics, observed_count=None, first_review_dbc=None):
    """Predict reviews in window (dbc_to, dbc_from], with optional scaling matching deployed pipeline."""
    pop_integral = model.population_prior.integrate_box_1d(dbc_to, dbc_from)
    expected = 0.0
    for _, row in model.profiles.df.iterrows():
        name = row['reviewer_name']
        if name in observed_critics:
            continue
        w = row['base_rate']
        entry = model.critic_kdes.get(name)
        if entry is None:
            continue
        integral = _blended_integral(
            entry, model.population_prior, dbc_to, dbc_from, pop_integral=pop_integral,
        )
        expected += w * integral
    if observed_count is not None and first_review_dbc is not None:
        scaling = _compute_scaling(model, dbc_from, observed_count, first_review_dbc)
        expected *= scaling
    return expected

def actual_in_window(target, dbc_from, dbc_to):
    target_close = close_date_map[target]
    movie_reviews = reviews[reviews['movie_slug'] == target].copy()
    movie_reviews['dbc'] = (target_close - movie_reviews['estimated_timestamp']).dt.total_seconds() / 86400
    return int(((movie_reviews['dbc'] > dbc_to) & (movie_reviews['dbc'] <= dbc_from)).sum())

# Cache key for KDE quality test results
KDE_CACHE_PATH = CACHE_DIR / 'kde_quality_test.pkl'

def run_kde_quality_test(force=False, verbose=True):
    """For each target, snap at T-3d and predict reviews in (T-3d, T-1d] using both methods."""
    if KDE_CACHE_PATH.exists() and not force:
        return pd.read_pickle(KDE_CACHE_PATH)

    rows = []
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue

        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=3)  # T-3d snap
        state = snapshot_state(target, snap_time)
        if state is None:
            continue

        for method, training in [
            ('baseline', default_training_slugs(
                movies, exclude_slug=target, before_date=close_date_map[target], n=20,
            )),
            ('stratified', matched_training_slugs(target, target_gap, band=3.0, n=20)[0]),
        ]:
            if len(training) < 5:
                continue
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model(profiles, verbose=False)

            pred = predict_window(
                model, dbc_from=3.0, dbc_to=1.0,
                observed_critics=state['observed_critics'],
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
            act = actual_in_window(target, dbc_from=3.0, dbc_to=1.0)
            rows.append({
                'target_slug': target,
                'target_gap': target_gap,
                'method': method,
                'predicted': pred,
                'actual': act,
            })

        if verbose and (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets processed')

    df = pd.DataFrame(rows)
    df.to_pickle(KDE_CACHE_PATH)
    print(f'Cached {len(df)} rows to {KDE_CACHE_PATH.name}')
    return df

quality = run_kde_quality_test()
quality.head(8)


In [ ]:
def kde_quality_summary(df, label=''):
    df = df.copy()
    df['err'] = df['predicted'] - df['actual']
    df['abs_err'] = df['err'].abs()

    rows = []
    for method in ['baseline', 'stratified']:
        m = df[df['method'] == method]
        rows.append({
            'subset': label,
            'method': method,
            'n': len(m),
            'MAE': m['abs_err'].mean(),
            'median_err': m['err'].median(),
            'mean_err': m['err'].mean(),
            'p90_abs_err': m['abs_err'].quantile(0.9),
        })
    return pd.DataFrame(rows)

# Full cohort
full_summary = kde_quality_summary(quality, label='full')

# Day-level-only subset
day_only = quality[~quality['target_slug'].isin(EXCLUDE_SLUGS)]
day_summary = kde_quality_summary(day_only, label='day-level only')

print('=== KDE quality test: predict (T-3d, T-1d] window ===')
print(pd.concat([full_summary, day_summary]).to_string(index=False, float_format='%.2f'))

print('\n--- Calibration interpretation ---')
for label, sub in [('full', quality), ('day-level only', day_only)]:
    for method in ['baseline', 'stratified']:
        m = sub[sub['method'] == method].copy()
        m['err'] = m['predicted'] - m['actual']
        sign = 'over-predicting' if m['err'].median() > 0 else 'under-predicting'
        print(f'  {label:15s} {method:10s}: median_err = {m["err"].median():+.2f}  ({sign})')


## 10. Shape vs scalar diagnostics

Decompose the +5-9 review over-prediction into shape bias vs scalar bias. If shape is right, all time regions are over-predicted by the same multiplicative factor — fixable with a single scalar correction. If shape is wrong, certain time regions are biased differently — needs structural fix.


### 10.1 Population prior shape comparison

Plot `population_prior(dbc)` for baseline and stratified training sets against the underlying training-data histograms. By construction the KDE should match its training data; verifying this is sanity. Differences between baseline and stratified pop priors show the effect of gap-matched selection on the aggregate shape.


In [ ]:
# Pick a reference target near the cohort median gap for representative comparison
ref_target = 'lilo_and_stitch_2025'
ref_gap = gap_for_slug(ref_target)
print(f'Reference target: {ref_target}  (gap = {ref_gap:.2f}d)')

base_training = default_training_slugs(
    movies, exclude_slug=ref_target, before_date=close_date_map[ref_target], n=20,
)
strat_training, eff_band = matched_training_slugs(ref_target, ref_gap, band=3.0, n=20)

base_profiles = build_critic_profiles(reviews, close_date_map, base_training, verbose=False)
base_model = build_kde_lambda_model(base_profiles, verbose=False)
strat_profiles = build_critic_profiles(reviews, close_date_map, strat_training, verbose=False)
strat_model = build_kde_lambda_model(strat_profiles, verbose=False)

# Pull all training timings for histogram overlay
def all_training_timings(profiles):
    return np.concatenate(profiles.df['timing_data'].values)

base_timings = all_training_timings(base_profiles)
strat_timings = all_training_timings(strat_profiles)

# Plot
t_grid = np.linspace(0, 14, 400)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, model, timings, label in [
    (axes[0], base_model, base_timings, f'baseline (n={len(base_timings)} reviews)'),
    (axes[1], strat_model, strat_timings, f'stratified band={eff_band:.1f}d (n={len(strat_timings)} reviews)'),
]:
    # Histogram (normalized to a density)
    ax.hist(timings[timings <= 14], bins=np.arange(0, 15, 0.5), density=True,
            alpha=0.4, color='gray', label='training data histogram')
    # KDE overlay
    ax.plot(t_grid, model.population_prior(t_grid), 'steelblue', lw=2, label='population prior KDE')
    ax.set_xlabel('Days before close')
    ax.set_ylabel('Density')
    ax.set_title(label)
    ax.invert_xaxis()
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle(f'Shape sanity check (target = {ref_target}, gap = {ref_gap:.1f}d)')
plt.tight_layout()


### 10.2 Per-window calibration test

Split (T-3d, T-1d] into two halves: (T-3d, T-2d] and (T-2d, T-1d]. For each target, predict each half separately and compute `predicted / actual`.

- If `median(ratio_early) ≈ median(ratio_late)`: bias is purely scalar (same factor in both regions). One multiplicative correction would fix it.
- If they differ: shape is wrong. The KDE puts mass in the wrong place.


In [ ]:
WINDOW_CACHE_PATH = CACHE_DIR / 'kde_window_split.pkl'

def run_window_split_test(force=False):
    """Per-target prediction in (T-3d, T-2d] and (T-2d, T-1d] half-windows."""
    if WINDOW_CACHE_PATH.exists() and not force:
        return pd.read_pickle(WINDOW_CACHE_PATH)

    rows = []
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue

        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=3)
        state = snapshot_state(target, snap_time)
        if state is None:
            continue

        for method, training in [
            ('baseline', default_training_slugs(
                movies, exclude_slug=target, before_date=close_date_map[target], n=20,
            )),
            ('stratified', matched_training_slugs(target, target_gap, band=3.0, n=20)[0]),
        ]:
            if len(training) < 5:
                continue
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model(profiles, verbose=False)

            for win_label, dbc_from, dbc_to in [
                ('early', 3.0, 2.0),  # (T-3d, T-2d]
                ('late', 2.0, 1.0),   # (T-2d, T-1d]
            ]:
                pred = predict_window(
                    model, dbc_from=dbc_from, dbc_to=dbc_to,
                    observed_critics=state['observed_critics'],
                    observed_count=state['observed_count'],
                    first_review_dbc=state['first_review_dbc'],
                )
                act = actual_in_window(target, dbc_from=dbc_from, dbc_to=dbc_to)
                rows.append({
                    'target_slug': target,
                    'target_gap': target_gap,
                    'method': method,
                    'window': win_label,
                    'predicted': pred,
                    'actual': act,
                })

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets processed')

    df = pd.DataFrame(rows)
    df.to_pickle(WINDOW_CACHE_PATH)
    print(f'Cached {len(df)} rows to {WINDOW_CACHE_PATH.name}')
    return df

window_df = run_window_split_test()
print()
print('First few rows:')
print(window_df.head(8).to_string(index=False))


In [ ]:
# Compute ratios and aggregate
window_df['ratio'] = window_df['predicted'] / window_df['actual']
# Drop divide-by-zero (actual=0)
clean = window_df[(window_df['actual'] > 0) & np.isfinite(window_df['ratio'])]

print('Per-window calibration (predicted / actual ratio):')
print()
agg = clean.groupby(['method', 'window'])['ratio'].agg(['median', 'mean', 'count']).round(3)
print(agg.to_string())

print()
print('--- Shape interpretation ---')
for method in ['baseline', 'stratified']:
    early_med = clean[(clean['method'] == method) & (clean['window'] == 'early')]['ratio'].median()
    late_med = clean[(clean['method'] == method) & (clean['window'] == 'late')]['ratio'].median()
    diff = early_med - late_med
    rel = (diff / late_med * 100) if late_med else 0
    verdict = 'shape OK (bias is scalar)' if abs(rel) < 15 else 'SHAPE BIASED'
    print(f'  {method:10s}: early/late = {early_med:.2f} / {late_med:.2f}  diff = {diff:+.2f} ({rel:+.1f}%)  -> {verdict}')

# Also report MAE per window for context
print()
print('Per-window MAE (predicted - actual abs):')
window_df['abs_err'] = (window_df['predicted'] - window_df['actual']).abs()
mae_agg = window_df.groupby(['method', 'window'])['abs_err'].mean().round(2)
print(mae_agg.to_string())


### 10.3 Per-target predicted/actual ratio distribution

If shape is OK from §10.2, the bias is purely scalar. This cell characterizes that scalar: distribution of `predicted / actual` per target. Median = systematic bias factor. Spread = how reliable that factor is per-target.


In [ ]:
# Use the existing KDE quality test results (already cached)
quality['ratio'] = quality['predicted'] / quality['actual']
clean_q = quality[(quality['actual'] > 0) & np.isfinite(quality['ratio'])]

print('Per-target predicted/actual ratio (full T-3d to T-1d window):')
print()
ratio_agg = clean_q.groupby('method')['ratio'].agg(['median', 'mean', 'std', 'count']).round(3)
print(ratio_agg.to_string())

# Plot ratio distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, method in zip(axes, ['baseline', 'stratified']):
    sub = clean_q[clean_q['method'] == method]['ratio']
    ax.hist(sub.clip(upper=8), bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(1.0, color='black', ls='--', alpha=0.5, label='ratio=1 (perfect)')
    ax.axvline(sub.median(), color='red', ls='-', alpha=0.7, label=f'median={sub.median():.2f}')
    ax.set_xlabel('predicted / actual (clipped at 8)')
    ax.set_ylabel('Count')
    ax.set_title(f'{method}: scalar bias distribution')
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()

# What scalar correction would fix it?
for method in ['baseline', 'stratified']:
    sub = clean_q[clean_q['method'] == method]['ratio']
    inv_med = 1.0 / sub.median()
    print(f'\n{method}: median ratio = {sub.median():.3f} -> scaling correction = {inv_med:.3f}')
    print(f'  Applied correction would shift median predicted/actual to ~1.0')


## 11. Relaxed scaling test

The over-prediction (median ratio = 1.91 baseline, 1.42 stratified) suggests `_compute_scaling`'s threshold (40) and lower clamp (0.5) are blocking aggressive enough corrections. Test two relaxed configs:

- **Default:** threshold=40, clamp=[0.5, 2.0] (sanity check, should match §10 results)
- **Relaxed:** threshold=10, clamp=[0.2, 2.0]
- **More relaxed:** threshold=5, clamp=[0.1, 2.0]

Inline custom `compute_scaling_custom()` and `predict_window_custom()` to avoid library changes. Test on the same (T-3d, T-1d] middle window so results are directly comparable to §9.


In [ ]:
def compute_scaling_custom(model, days_before_close, observed_count, first_review_dbc,
                            threshold=40.0, clamp=(0.5, 2.0)):
    pop_integral = model.population_prior.integrate_box_1d(days_before_close, first_review_dbc)
    expected_so_far = 0.0
    for _, row in model.profiles.df.iterrows():
        w = row['base_rate']
        entry = model.critic_kdes.get(row['reviewer_name'])
        if entry is None:
            continue
        integral = _blended_integral(
            entry, model.population_prior, days_before_close, first_review_dbc,
            pop_integral=pop_integral,
        )
        expected_so_far += w * integral
    if expected_so_far < threshold:
        return 1.0, expected_so_far, False  # scaling skipped
    scaling = observed_count / expected_so_far
    clamped = max(clamp[0], min(clamp[1], scaling))
    return clamped, expected_so_far, True

def predict_window_custom(model, dbc_from, dbc_to, observed_critics,
                          observed_count=None, first_review_dbc=None,
                          threshold=40.0, clamp=(0.5, 2.0)):
    pop_integral = model.population_prior.integrate_box_1d(dbc_to, dbc_from)
    expected = 0.0
    for _, row in model.profiles.df.iterrows():
        name = row['reviewer_name']
        if name in observed_critics:
            continue
        w = row['base_rate']
        entry = model.critic_kdes.get(name)
        if entry is None:
            continue
        integral = _blended_integral(
            entry, model.population_prior, dbc_to, dbc_from, pop_integral=pop_integral,
        )
        expected += w * integral
    scaling, expected_so_far, fired = 1.0, 0.0, False
    if observed_count is not None and first_review_dbc is not None:
        scaling, expected_so_far, fired = compute_scaling_custom(
            model, dbc_from, observed_count, first_review_dbc,
            threshold=threshold, clamp=clamp,
        )
    return expected * scaling, scaling, expected_so_far, fired

RELAXED_CACHE_PATH = CACHE_DIR / 'kde_relaxed_scaling.pkl'

def run_relaxed_test(threshold, clamp, force=False):
    config_key = f'thr={threshold:g}_clamp=[{clamp[0]:g},{clamp[1]:g}]'
    if RELAXED_CACHE_PATH.exists() and not force:
        cached = pd.read_pickle(RELAXED_CACHE_PATH)
        if 'config' in cached.columns and (cached['config'] == config_key).any():
            print(f'Loading cached results for {config_key}')
            return cached[cached['config'] == config_key].copy()
    else:
        cached = pd.DataFrame()

    print(f'Running {config_key}...')
    rows = []
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=3)
        state = snapshot_state(target, snap_time)
        if state is None:
            continue

        for method, training in [
            ('baseline', default_training_slugs(
                movies, exclude_slug=target, before_date=close_date_map[target], n=20,
            )),
            ('stratified', matched_training_slugs(target, target_gap, band=3.0, n=20)[0]),
        ]:
            if len(training) < 5:
                continue
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model(profiles, verbose=False)

            pred, scaling, exp_so_far, fired = predict_window_custom(
                model, dbc_from=3.0, dbc_to=1.0,
                observed_critics=state['observed_critics'],
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
                threshold=threshold, clamp=clamp,
            )
            act = actual_in_window(target, dbc_from=3.0, dbc_to=1.0)
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'method': method, 'predicted': pred, 'actual': act,
                'scaling': scaling, 'expected_so_far': exp_so_far,
                'scaling_fired': fired, 'config': config_key,
            })

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df = pd.concat([cached, df], ignore_index=True)
        df = df.drop_duplicates(['target_slug', 'method', 'config'], keep='last')
    df.to_pickle(RELAXED_CACHE_PATH)
    return df[df['config'] == config_key].copy()

# Run three configs
configs = [
    (40.0, (0.5, 2.0)),  # default
    (10.0, (0.2, 2.0)),  # relaxed
    (5.0, (0.1, 2.0)),   # more relaxed
]
all_results = []
for thr, clamp in configs:
    res = run_relaxed_test(thr, clamp)
    all_results.append(res)
combined = pd.concat(all_results, ignore_index=True)
print(f'\nTotal rows across configs: {len(combined)}')


In [ ]:
# Aggregate: median ratio, MAE, scaling fire rate per (config, method)
combined['ratio'] = combined['predicted'] / combined['actual']
combined['err'] = combined['predicted'] - combined['actual']
combined['abs_err'] = combined['err'].abs()
clean = combined[(combined['actual'] > 0) & np.isfinite(combined['ratio'])]

print('Per-config calibration on (T-3d, T-1d] window:')
print()
agg = clean.groupby(['config', 'method']).agg(
    n=('target_slug', 'count'),
    median_ratio=('ratio', 'median'),
    mean_ratio=('ratio', 'mean'),
    MAE=('abs_err', 'mean'),
    median_err=('err', 'median'),
    scaling_fire_rate=('scaling_fired', 'mean'),
    median_scaling=('scaling', 'median'),
).round(3)
print(agg.to_string())

print('\n--- Summary: how does relaxation change median ratio? ---')
for method in ['baseline', 'stratified']:
    print(f'\n{method}:')
    for thr, clamp in configs:
        config_key = f'thr={thr:g}_clamp=[{clamp[0]:g},{clamp[1]:g}]'
        sub = clean[(clean['config'] == config_key) & (clean['method'] == method)]
        ratio = sub['ratio'].median()
        mae = sub['abs_err'].mean()
        fired = sub['scaling_fired'].mean() * 100
        print(f'  {config_key:35s}  median_ratio={ratio:.2f}  MAE={mae:.2f}  scaling_fired={fired:.0f}%')


In [ ]:
# Distribution of scaling factors when fired (under each config)
print('Distribution of scaling factor (when fired):')
for thr, clamp in configs:
    config_key = f'thr={thr:g}_clamp=[{clamp[0]:g},{clamp[1]:g}]'
    for method in ['baseline', 'stratified']:
        sub = combined[
            (combined['config'] == config_key)
            & (combined['method'] == method)
            & (combined['scaling_fired'])
        ]
        if len(sub) == 0:
            continue
        print(f'  {method:10s} {config_key}:')
        print(f'    n_fired={len(sub)}  scaling: median={sub["scaling"].median():.2f}  '
              f'p25={sub["scaling"].quantile(0.25):.2f}  p75={sub["scaling"].quantile(0.75):.2f}')
        # How often did scaling hit a clamp boundary?
        at_lower = (sub['scaling'] == clamp[0]).mean() * 100
        at_upper = (sub['scaling'] == clamp[1]).mean() * 100
        print(f'    pinned at lower clamp ({clamp[0]}): {at_lower:.0f}%  upper clamp ({clamp[1]}): {at_upper:.0f}%')


## 12. Bandwidth cap test

Hypothesis: Scott's rule produces effective bandwidths of 1.5-3d for sparse, spread-out training data, smoothing each review across multiple days. For day-level data, this over-smooths — the spike near embargo lift gets blurred into a wide hump, which the model under-predicts. Then `_compute_scaling` over-extrapolates that under-estimate.

Test: cap bandwidth at 1.0d (in addition to the 0.5d floor). For day-level data, this constrains kernels to ~within-day spread, no bleeding into adjacent days.

Inline custom KDE builder to avoid library changes.


In [ ]:
from scipy.stats import gaussian_kde

def _fit_critic_kde_capped(timing_data, population_prior, shrinkage_k, bandwidth_floor, bandwidth_ceiling):
    """Like _fit_critic_kde but with a bandwidth ceiling on top of the floor."""
    n = len(timing_data)
    result = {'empirical': None, 'n': n, 'k': shrinkage_k}
    if n < 2:
        return result
    if timing_data.std() == 0:
        return result
    try:
        kde = gaussian_kde(timing_data)
        effective_bw = kde.factor * timing_data.std()
        if effective_bw < bandwidth_floor:
            kde.set_bandwidth(bandwidth_floor / timing_data.std())
        elif effective_bw > bandwidth_ceiling:
            kde.set_bandwidth(bandwidth_ceiling / timing_data.std())
        result['empirical'] = kde
    except np.linalg.LinAlgError:
        pass
    return result

def build_kde_lambda_model_capped(profiles, shrinkage_k=3.0, bandwidth_floor=0.5, bandwidth_ceiling=1.0):
    """Like build_kde_lambda_model but with a bandwidth ceiling."""
    from rotten_tomatoes_forecasting.critic_model import KDELambdaModel
    all_timing = np.concatenate(profiles.df['timing_data'].values)
    population_prior = gaussian_kde(all_timing)
    pop_bw = population_prior.factor * all_timing.std()
    if pop_bw > bandwidth_ceiling:
        population_prior.set_bandwidth(bandwidth_ceiling / all_timing.std())

    critic_kdes = {}
    for _, row in profiles.df.iterrows():
        timing = np.array(row['timing_data'])
        entry = _fit_critic_kde_capped(
            timing, population_prior, shrinkage_k, bandwidth_floor, bandwidth_ceiling,
        )
        critic_kdes[row['reviewer_name']] = entry

    return KDELambdaModel(
        profiles=profiles,
        population_prior=population_prior,
        critic_kdes=critic_kdes,
        shrinkage_k=shrinkage_k,
        bandwidth_floor=bandwidth_floor,
    )


In [ ]:
# Diagnostic: how often is the ceiling actually binding under default Scott's rule?
sample_target = 'lilo_and_stitch_2025'
sample_gap = gap_for_slug(sample_target)
sample_training = matched_training_slugs(sample_target, sample_gap, band=3.0, n=20)[0]
sample_profiles = build_critic_profiles(reviews, close_date_map, sample_training, verbose=False)

bw_stats = []
for _, row in sample_profiles.df.iterrows():
    timing = np.array(row['timing_data'])
    if len(timing) < 2 or timing.std() == 0:
        continue
    kde = gaussian_kde(timing)
    effective_bw = kde.factor * timing.std()
    bw_stats.append({
        'critic': row['reviewer_name'],
        'n_reviews': len(timing),
        'std': timing.std(),
        'scott_bw': effective_bw,
    })

bw_df = pd.DataFrame(bw_stats)
print(f'Reference target: {sample_target} (gap = {sample_gap:.2f}d)')
print(f'Critics with empirical KDE: {len(bw_df)}')
print()
print('Scott\'s rule effective bandwidth distribution:')
print(bw_df['scott_bw'].describe([.25, .5, .75, .9, .95]).round(3).to_string())
print()
for ceiling in [0.7, 1.0, 1.5]:
    pct_above = (bw_df['scott_bw'] > ceiling).mean() * 100
    print(f'  Fraction of critics with Scott\'s bw > {ceiling}d: {pct_above:.1f}%')


In [ ]:
BANDWIDTH_CACHE_PATH = CACHE_DIR / 'kde_bandwidth_cap.pkl'

def run_bandwidth_test(bandwidth_ceiling, force=False):
    cache_key = f'ceil={bandwidth_ceiling:g}'
    if BANDWIDTH_CACHE_PATH.exists() and not force:
        cached = pd.read_pickle(BANDWIDTH_CACHE_PATH)
        if 'config' in cached.columns and (cached['config'] == cache_key).any():
            print(f'Loading cached results for {cache_key}')
            return cached[cached['config'] == cache_key].copy()
    else:
        cached = pd.DataFrame()

    print(f'Running {cache_key}...')
    rows = []
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=3)
        state = snapshot_state(target, snap_time)
        if state is None:
            continue

        for method, training in [
            ('baseline', default_training_slugs(
                movies, exclude_slug=target, before_date=close_date_map[target], n=20,
            )),
            ('stratified', matched_training_slugs(target, target_gap, band=3.0, n=20)[0]),
        ]:
            if len(training) < 5:
                continue
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model_capped(
                profiles, bandwidth_floor=0.5, bandwidth_ceiling=bandwidth_ceiling,
            )
            pred = predict_window(
                model, dbc_from=3.0, dbc_to=1.0,
                observed_critics=state['observed_critics'],
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
            act = actual_in_window(target, dbc_from=3.0, dbc_to=1.0)
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'method': method, 'predicted': pred, 'actual': act,
                'config': cache_key,
            })
        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df = pd.concat([cached, df], ignore_index=True)
        df = df.drop_duplicates(['target_slug', 'method', 'config'], keep='last')
    df.to_pickle(BANDWIDTH_CACHE_PATH)
    return df[df['config'] == cache_key].copy()

# Run multiple ceilings
ceilings = [1.5, 1.0, 0.7]
all_bw_results = []
for c in ceilings:
    res = run_bandwidth_test(c)
    all_bw_results.append(res)
bw_combined = pd.concat(all_bw_results, ignore_index=True)
print(f'\nTotal rows: {len(bw_combined)}')


In [ ]:
# Compare against the no-cap baseline (from §9 KDE quality test)
no_cap = quality.copy()
no_cap['config'] = 'no_cap (Scott)'

bw_combined['ratio'] = bw_combined['predicted'] / bw_combined['actual']
bw_combined['err'] = bw_combined['predicted'] - bw_combined['actual']
bw_combined['abs_err'] = bw_combined['err'].abs()
no_cap['ratio'] = no_cap['predicted'] / no_cap['actual']
no_cap['err'] = no_cap['predicted'] - no_cap['actual']
no_cap['abs_err'] = no_cap['err'].abs()

all_bw = pd.concat([no_cap, bw_combined], ignore_index=True)
clean_bw = all_bw[(all_bw['actual'] > 0) & np.isfinite(all_bw['ratio'])]

print('Bandwidth cap comparison on (T-3d, T-1d] window:')
print()
agg = clean_bw.groupby(['config', 'method']).agg(
    n=('target_slug', 'count'),
    median_ratio=('ratio', 'median'),
    mean_ratio=('ratio', 'mean'),
    MAE=('abs_err', 'mean'),
    median_err=('err', 'median'),
).round(3)
print(agg.to_string())

print('\n--- Quick comparison ---')
for method in ['baseline', 'stratified']:
    print(f'\n{method}:')
    for cfg in ['no_cap (Scott)', 'ceil=1.5', 'ceil=1', 'ceil=0.7']:
        sub = clean_bw[(clean_bw['config'] == cfg) & (clean_bw['method'] == method)]
        if len(sub) == 0:
            continue
        print(f'  {cfg:18s}  median_ratio={sub["ratio"].median():.2f}  '
              f'MAE={sub["abs_err"].mean():.2f}  median_err={sub["err"].median():+.2f}')


## 13. Critic-overlap similarity test (Phase A)

Following plan: `plans/plan_critic_overlap_similarity.md`. Tests whether adding **critic-overlap** as a similarity dimension on top of gap improves predictions vs the current best (`stratified + ceil=0.7`, MAE=5.57 on (T-3d, T-1d] middle window).

**Methods compared:**

1. `stratified + ceil=0.7` (control) — current best.
2. `gap_overlap_ranked + ceil=0.7` — filter `|gap_diff| ≤ 5d`, rank by Jaccard, top 20.
3. `combined_score + ceil=0.7` — weighted score `α · exp(−|gap_diff|/8) + (1−α) · jaccard`, top 20. α swept.
4. `gap_overlap_ranked + no_cap` — bandwidth-cap ablation.

**Snaps:** T-3d (primary), T-1d (secondary directional check).

**Skip rules:** target requires `first_review_dbc ≥ 4d` and `≥3 observed critics`.

**Decision rule:** point estimate of MAE improvement ≥10% AND lower bound of 95% bootstrap CI > 0 → promote.


In [ ]:
# Helpers
def critics_in_window(slug, window_start, window_days):
    """Return set of critics who reviewed `slug` within [window_start, window_start + window_days]."""
    window_end = window_start + pd.Timedelta(days=window_days)
    movie_reviews = reviews[
        (reviews['movie_slug'] == slug)
        & (reviews['estimated_timestamp'] >= window_start)
        & (reviews['estimated_timestamp'] <= window_end)
    ]
    return set(movie_reviews['reviewer_name'])

def jaccard(set_a, set_b):
    union = set_a | set_b
    if not union:
        return 0.0
    return len(set_a & set_b) / len(union)

# Pre-compute first_review_ts per movie (already in `first_review_ts` Series from §1)
print(f'first_review_ts available for {len(first_review_ts)} movies')

# Spot-check critic_set extraction
sample_slug = 'lilo_and_stitch_2025'
sample_first = first_review_ts.loc[sample_slug]
sample_critics_4d = critics_in_window(sample_slug, sample_first, window_days=4.0)
print(f'\n{sample_slug}: critics in first 4d post-first-review = {len(sample_critics_4d)}')
print(f'  sample: {list(sample_critics_4d)[:5]}')


In [ ]:
# Selector functions

def gap_overlap_ranked_selector(target, target_gap, target_critics, target_window_days,
                                 k=20, gap_band=5.0):
    """Filter to |gap_diff| ≤ gap_band, rank by Jaccard against target_critics, top k."""
    target_close = close_date_map[target]
    candidates = gaps[
        (gaps['close_ts'] < target_close)
        & (gaps['slug'] != target)
        & ((gaps['gap_days'] - target_gap).abs() <= gap_band)
    ]
    if len(candidates) == 0:
        return [], 0.0
    rows = []
    for _, row in candidates.iterrows():
        slug = row['slug']
        train_first = first_review_ts.loc[slug]
        train_critics = critics_in_window(slug, train_first, target_window_days)
        score = jaccard(target_critics, train_critics)
        rows.append((slug, score))
    rows.sort(key=lambda x: x[1], reverse=True)
    selected = [r[0] for r in rows[:k]]
    median_score = np.median([r[1] for r in rows[:k]]) if rows else 0.0
    return selected, float(median_score)

def combined_score_selector(target, target_gap, target_critics, target_window_days,
                            k=20, alpha=0.5, sigma_gap=8.0):
    """Weighted score over all candidates; top k."""
    target_close = close_date_map[target]
    candidates = gaps[
        (gaps['close_ts'] < target_close)
        & (gaps['slug'] != target)
    ]
    if len(candidates) == 0:
        return [], 0.0
    rows = []
    for _, row in candidates.iterrows():
        slug = row['slug']
        gap_diff = abs(row['gap_days'] - target_gap)
        gap_score = float(np.exp(-gap_diff / sigma_gap))
        train_first = first_review_ts.loc[slug]
        train_critics = critics_in_window(slug, train_first, target_window_days)
        j = jaccard(target_critics, train_critics)
        combined = alpha * gap_score + (1 - alpha) * j
        rows.append((slug, combined, gap_score, j))
    rows.sort(key=lambda x: x[1], reverse=True)
    selected = [r[0] for r in rows[:k]]
    median_score = np.median([r[1] for r in rows[:k]]) if rows else 0.0
    return selected, float(median_score)

# Spot-check on a sample target
sample_target = 'lilo_and_stitch_2025'
sample_gap = gap_for_slug(sample_target)
sample_close = close_date_map[sample_target]
sample_snap_time = sample_close - pd.Timedelta(days=3)
sample_state = snapshot_state(sample_target, sample_snap_time)
sample_first_dbc = sample_state['first_review_dbc']
sample_window_days = sample_first_dbc - 3.0  # days from first review to T-3d snap
print(f'Target: {sample_target} (gap={sample_gap:.2f}d, first_review_dbc={sample_first_dbc:.2f}d)')
print(f'  observed critics at T-3d: {len(sample_state["observed_critics"])}')
print(f'  comparison window: {sample_window_days:.2f}d post-first-review')

selected_g_o, med_score_g_o = gap_overlap_ranked_selector(
    sample_target, sample_gap, sample_state['observed_critics'], sample_window_days,
)
print(f'\ngap_overlap_ranked picked {len(selected_g_o)} movies, median jaccard score={med_score_g_o:.3f}')
print(f'  first 5: {selected_g_o[:5]}')

selected_combo, med_score_combo = combined_score_selector(
    sample_target, sample_gap, sample_state['observed_critics'], sample_window_days, alpha=0.5,
)
print(f'\ncombined_score (alpha=0.5) picked {len(selected_combo)} movies, median combined_score={med_score_combo:.3f}')
print(f'  first 5: {selected_combo[:5]}')


In [ ]:
# LOO loop with skip rules
OVERLAP_CACHE_PATH = CACHE_DIR / 'critic_overlap_test.pkl'

def passes_skip_rules(state, snap_dbc, min_first_review_dbc=4.0, min_critics=3):
    if state is None:
        return False, 'no observations'
    if state['first_review_dbc'] < min_first_review_dbc:
        return False, f'first_review_dbc={state["first_review_dbc"]:.2f} < {min_first_review_dbc}'
    if len(state['observed_critics']) < min_critics:
        return False, f'critics={len(state["observed_critics"])} < {min_critics}'
    return True, None

def run_overlap_test(snap_dbc, alpha=0.5, force=False):
    """Run all 4 methods at given snap. Predicts in window from snap to snap-2d (or to dbc=1 if snap=1)."""
    # Window definition: middle window for T-3d snap is (T-3d, T-1d]; for T-1d snap is (T-1d, dbc=0]
    if snap_dbc == 3.0:
        window_dbc_to = 1.0
    elif snap_dbc == 1.0:
        window_dbc_to = 0.01  # very thin slice between T-1d and midnight UTC of close
    else:
        raise ValueError(f'unsupported snap_dbc={snap_dbc}')

    config_key = f'snap={snap_dbc:g}_alpha={alpha:g}'
    cached = pd.read_pickle(OVERLAP_CACHE_PATH) if OVERLAP_CACHE_PATH.exists() else pd.DataFrame()
    if not force and not cached.empty and 'config' in cached.columns:
        if (cached['config'] == config_key).any():
            print(f'Loading cached results for {config_key}')
            return cached[cached['config'] == config_key].copy()

    print(f'Running {config_key}...')
    rows = []
    skip_log = {'no_obs': 0, 'low_first_review_dbc': 0, 'low_critics': 0}
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=snap_dbc)
        state = snapshot_state(target, snap_time)
        passed, reason = passes_skip_rules(state, snap_dbc)
        if not passed:
            if reason and 'no obs' in reason:
                skip_log['no_obs'] += 1
            elif 'first_review_dbc' in (reason or ''):
                skip_log['low_first_review_dbc'] += 1
            else:
                skip_log['low_critics'] += 1
            continue

        target_window_days = state['first_review_dbc'] - snap_dbc
        target_critics = state['observed_critics']

        method_specs = [
            ('control_stratified', lambda: matched_training_slugs(target, target_gap, band=3.0, n=20)[0], True),
            ('gap_overlap_ranked', lambda: gap_overlap_ranked_selector(
                target, target_gap, target_critics, target_window_days, k=20, gap_band=5.0,
            )[0], True),
            ('combined_score', lambda: combined_score_selector(
                target, target_gap, target_critics, target_window_days, k=20, alpha=alpha, sigma_gap=8.0,
            )[0], True),
            ('gap_overlap_no_cap', lambda: gap_overlap_ranked_selector(
                target, target_gap, target_critics, target_window_days, k=20, gap_band=5.0,
            )[0], False),  # no bandwidth cap
        ]

        for method_name, training_fn, use_cap in method_specs:
            training = training_fn()
            if len(training) < 5:
                rows.append({
                    'target_slug': target, 'target_gap': target_gap,
                    'snap_dbc': snap_dbc, 'method': method_name,
                    'predicted': np.nan, 'actual': np.nan,
                    'config': config_key,
                })
                continue
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            if use_cap:
                model = build_kde_lambda_model_capped(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
            else:
                model = build_kde_lambda_model(profiles, verbose=False)

            pred = predict_window(
                model, dbc_from=snap_dbc, dbc_to=window_dbc_to,
                observed_critics=state['observed_critics'],
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
            act = actual_in_window(target, dbc_from=snap_dbc, dbc_to=window_dbc_to)
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'snap_dbc': snap_dbc, 'method': method_name,
                'predicted': pred, 'actual': act,
                'config': config_key,
            })

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    print(f'  Skipped: no_obs={skip_log["no_obs"]}  low_first_review_dbc={skip_log["low_first_review_dbc"]}  low_critics={skip_log["low_critics"]}')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df_combined = pd.concat([cached, df], ignore_index=True)
        df_combined = df_combined.drop_duplicates(['target_slug', 'snap_dbc', 'method', 'config'], keep='last')
    else:
        df_combined = df
    df_combined.to_pickle(OVERLAP_CACHE_PATH)
    return df_combined[df_combined['config'] == config_key].copy()

# Run T-3d and T-1d at default alpha=0.5
results_t3 = run_overlap_test(snap_dbc=3.0, alpha=0.5)
results_t1 = run_overlap_test(snap_dbc=1.0, alpha=0.5)
print(f'\nTotal rows T-3d: {len(results_t3)}, T-1d: {len(results_t1)}')


In [ ]:
# Aggregate, intersect, bootstrap CI
def bootstrap_mae_delta(deltas, n_boot=1000, seed=42):
    """Return (point_estimate, lower_95, upper_95) of mean of deltas."""
    rng = np.random.default_rng(seed)
    n = len(deltas)
    if n == 0:
        return np.nan, np.nan, np.nan
    boot_means = np.array([
        rng.choice(deltas, size=n, replace=True).mean()
        for _ in range(n_boot)
    ])
    return float(np.mean(deltas)), float(np.quantile(boot_means, 0.025)), float(np.quantile(boot_means, 0.975))

def summarize(results, snap_label, control='control_stratified'):
    df = results.dropna(subset=['predicted', 'actual']).copy()
    df['err'] = df['predicted'] - df['actual']
    df['abs_err'] = df['err'].abs()
    df['ratio'] = df['predicted'] / df['actual']

    methods = ['control_stratified', 'gap_overlap_ranked', 'combined_score', 'gap_overlap_no_cap']
    common_targets = (
        df.groupby('target_slug')['method'].nunique() == len(methods)
    )
    common_targets = common_targets[common_targets].index
    df = df[df['target_slug'].isin(common_targets)]

    print(f'\n=== {snap_label} | n_common = {len(common_targets)} ===')
    rows = []
    for method in methods:
        sub = df[df['method'] == method]
        clean = sub[sub['actual'] > 0]
        rows.append({
            'method': method,
            'n': len(sub),
            'MAE': sub['abs_err'].mean(),
            'median_err': sub['err'].median(),
            'median_ratio': clean['ratio'].median() if len(clean) else np.nan,
        })
    summary_df = pd.DataFrame(rows)
    print(summary_df.to_string(index=False, float_format='%.3f'))

    # Bootstrap CI on MAE delta vs control
    print(f'\n--- Bootstrap 95% CI on MAE delta (vs {control}) ---')
    control_errs = df[df['method'] == control].set_index('target_slug')['abs_err']
    for method in methods:
        if method == control:
            continue
        method_errs = df[df['method'] == method].set_index('target_slug')['abs_err']
        common_idx = control_errs.index.intersection(method_errs.index)
        deltas = (control_errs.loc[common_idx] - method_errs.loc[common_idx]).values  # positive = better
        point, lo, hi = bootstrap_mae_delta(deltas)
        rel = point / control_errs.loc[common_idx].mean() * 100
        sig = 'SIG' if lo > 0 else 'ns'
        print(f'  {method:25s}  delta = {point:+.3f}  ({rel:+5.1f}%)  CI95 = [{lo:+.3f}, {hi:+.3f}]  {sig}')

summarize(results_t3, 'T-3d snap, predict (T-3d, T-1d]')
summarize(results_t1, 'T-1d snap, predict (T-1d, ~midnight UTC of close]')


In [ ]:
# Alpha sweep for combined_score
print('=== Alpha sweep for combined_score (T-3d snap) ===')
alpha_results = {}
for a in [0.1, 0.3, 0.5, 0.7, 0.9]:
    res = run_overlap_test(snap_dbc=3.0, alpha=a)
    alpha_results[a] = res

for a, res in alpha_results.items():
    df = res.dropna(subset=['predicted', 'actual']).copy()
    df = df[df['method'] == 'combined_score']
    df['abs_err'] = (df['predicted'] - df['actual']).abs()
    df['ratio'] = df['predicted'] / df['actual']
    clean = df[df['actual'] > 0]
    print(f'  alpha={a}:  MAE={df["abs_err"].mean():.3f}  median_ratio={clean["ratio"].median():.3f}  n={len(df)}')


In [ ]:
# By gap quantile breakdown (T-3d primary)
def by_quantile(results, snap_label):
    df = results.dropna(subset=['predicted', 'actual']).copy()
    df['abs_err'] = (df['predicted'] - df['actual']).abs()

    def qbin(g):
        if g <= q_cutoffs[0]: return 'Q1'
        if g <= q_cutoffs[1]: return 'Q2'
        if g <= q_cutoffs[2]: return 'Q3'
        return 'Q4'
    df['quantile'] = df['target_gap'].apply(qbin)

    methods = ['control_stratified', 'gap_overlap_ranked', 'combined_score', 'gap_overlap_no_cap']
    common = df.groupby('target_slug')['method'].nunique() == len(methods)
    df = df[df['target_slug'].isin(common[common].index)]

    print(f'\n=== {snap_label} | MAE by gap quantile ===')
    rows = []
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        qdf = df[df['quantile'] == q]
        row = {'quantile': q, 'n': qdf['target_slug'].nunique()}
        for m in methods:
            short = m.replace('control_', '').replace('gap_overlap_', 'g+o_').replace('combined_score', 'combo')
            row[short] = qdf[qdf['method'] == m]['abs_err'].mean()
        rows.append(row)
    print(pd.DataFrame(rows).to_string(index=False, float_format='%.2f'))

by_quantile(results_t3, 'T-3d snap')
by_quantile(results_t1, 'T-1d snap')


### 13.1 Phase A re-run: full snap-to-midnight-UTC window

Per Jake's clarification: predict the full window from snap to midnight UTC of close day (excludes close-day reviews via library's `dbc > 0` filter, but includes the day-before-close that the §9 middle-window test had cut off).

This is the deployment-relevant convention (matches sections 3-5 baseline). Numbers are NOT directly comparable to §10 (different window).


In [ ]:
OVERLAP_FULL_CACHE_PATH = CACHE_DIR / 'critic_overlap_test_full_window.pkl'

def run_overlap_test_full(snap_dbc, alpha=0.5, force=False):
    """Same as run_overlap_test but predicts the full snap-to-midnight-UTC window (dbc_to=0)."""
    config_key = f'snap={snap_dbc:g}_alpha={alpha:g}_full'
    cached = pd.read_pickle(OVERLAP_FULL_CACHE_PATH) if OVERLAP_FULL_CACHE_PATH.exists() else pd.DataFrame()
    if not force and not cached.empty and 'config' in cached.columns:
        if (cached['config'] == config_key).any():
            print(f'Loading cached results for {config_key}')
            return cached[cached['config'] == config_key].copy()

    print(f'Running {config_key}...')
    rows = []
    skip_log = {'no_obs': 0, 'low_first_review_dbc': 0, 'low_critics': 0}
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=snap_dbc)
        state = snapshot_state(target, snap_time)
        passed, reason = passes_skip_rules(state, snap_dbc)
        if not passed:
            if reason and 'no obs' in reason:
                skip_log['no_obs'] += 1
            elif 'first_review_dbc' in (reason or ''):
                skip_log['low_first_review_dbc'] += 1
            else:
                skip_log['low_critics'] += 1
            continue

        target_window_days = state['first_review_dbc'] - snap_dbc
        target_critics = state['observed_critics']

        method_specs = [
            ('control_stratified', lambda: matched_training_slugs(target, target_gap, band=3.0, n=20)[0], True),
            ('gap_overlap_ranked', lambda: gap_overlap_ranked_selector(
                target, target_gap, target_critics, target_window_days, k=20, gap_band=5.0,
            )[0], True),
            ('combined_score', lambda: combined_score_selector(
                target, target_gap, target_critics, target_window_days, k=20, alpha=alpha, sigma_gap=8.0,
            )[0], True),
        ]

        for method_name, training_fn, use_cap in method_specs:
            training = training_fn()
            if len(training) < 5:
                rows.append({
                    'target_slug': target, 'target_gap': target_gap,
                    'snap_dbc': snap_dbc, 'method': method_name,
                    'predicted': np.nan, 'actual': np.nan,
                    'config': config_key,
                })
                continue
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model_capped(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)

            # Full snap-to-midnight-UTC window: dbc_to=0
            pred = predict_window(
                model, dbc_from=snap_dbc, dbc_to=0.0,
                observed_critics=state['observed_critics'],
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
            # Match library's actual_remaining filter: 0 < dbc <= snap_dbc
            act = actual_remaining(target, snap_dbc)
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'snap_dbc': snap_dbc, 'method': method_name,
                'predicted': pred, 'actual': act,
                'config': config_key,
            })

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    print(f'  Skipped: {skip_log}')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df_combined = pd.concat([cached, df], ignore_index=True)
        df_combined = df_combined.drop_duplicates(['target_slug', 'snap_dbc', 'method', 'config'], keep='last')
    else:
        df_combined = df
    df_combined.to_pickle(OVERLAP_FULL_CACHE_PATH)
    return df_combined[df_combined['config'] == config_key].copy()

results_t3_full = run_overlap_test_full(snap_dbc=3.0, alpha=0.5)
results_t1_full = run_overlap_test_full(snap_dbc=1.0, alpha=0.5)
print(f'\nTotal rows T-3d: {len(results_t3_full)}, T-1d: {len(results_t1_full)}')


In [ ]:
# Reuse the same summarize() function from cell 54
def summarize_full(results, snap_label, control='control_stratified'):
    df = results.dropna(subset=['predicted', 'actual']).copy()
    df['err'] = df['predicted'] - df['actual']
    df['abs_err'] = df['err'].abs()
    df['ratio'] = df['predicted'] / df['actual']

    methods = ['control_stratified', 'gap_overlap_ranked', 'combined_score']
    common_targets = (
        df.groupby('target_slug')['method'].nunique() == len(methods)
    )
    common_targets = common_targets[common_targets].index
    df = df[df['target_slug'].isin(common_targets)]

    print(f'\n=== {snap_label} | n_common = {len(common_targets)} ===')
    rows = []
    for method in methods:
        sub = df[df['method'] == method]
        clean = sub[sub['actual'] > 0]
        rows.append({
            'method': method,
            'n': len(sub),
            'MAE': sub['abs_err'].mean(),
            'median_err': sub['err'].median(),
            'median_ratio': clean['ratio'].median() if len(clean) else np.nan,
        })
    summary_df = pd.DataFrame(rows)
    print(summary_df.to_string(index=False, float_format='%.3f'))

    print(f'\n--- Bootstrap 95% CI on MAE delta (vs {control}) ---')
    control_errs = df[df['method'] == control].set_index('target_slug')['abs_err']
    for method in methods:
        if method == control:
            continue
        method_errs = df[df['method'] == method].set_index('target_slug')['abs_err']
        common_idx = control_errs.index.intersection(method_errs.index)
        deltas = (control_errs.loc[common_idx] - method_errs.loc[common_idx]).values
        point, lo, hi = bootstrap_mae_delta(deltas)
        rel = point / control_errs.loc[common_idx].mean() * 100
        sig = 'SIG' if lo > 0 else 'ns'
        print(f'  {method:25s}  delta = {point:+.3f}  ({rel:+5.1f}%)  CI95 = [{lo:+.3f}, {hi:+.3f}]  {sig}')

summarize_full(results_t3_full, 'T-3d snap, predict (T-3d, midnight UTC of close]')
summarize_full(results_t1_full, 'T-1d snap, predict (T-1d, midnight UTC of close]')


### 13.2 Phase A re-run at T-5d snap (full window)

Per Jake's request: extend the test to T-5d snap. Predicts the full 5-day window from snap to midnight UTC of close. Tests whether Phase A's signal holds at earlier decision points (less observed data, longer prediction horizon).

**Caveat:** at T-5d snap, targets with `first_review_dbc < 5d` have zero observed reviews — will be skipped. Cohort first_review_dbc Q1=5.62d, so ~25-30% skip rate expected. Remaining sample skews toward Q3-Q4 (longer-gap targets).


In [ ]:
# Need to relax skip rule for T-5d (first_review_dbc must be >= snap_dbc + buffer)
def passes_skip_rules_for_snap(state, snap_dbc, min_first_review_dbc=None, min_critics=3):
    if state is None:
        return False, 'no observations'
    # Default min_first_review_dbc adapts to snap
    if min_first_review_dbc is None:
        min_first_review_dbc = snap_dbc + 1.0  # snap must be at least 1 day after first review
    if state['first_review_dbc'] < min_first_review_dbc:
        return False, f'first_review_dbc={state["first_review_dbc"]:.2f} < {min_first_review_dbc}'
    if len(state['observed_critics']) < min_critics:
        return False, f'critics={len(state["observed_critics"])} < {min_critics}'
    return True, None

def run_overlap_test_full_v2(snap_dbc, alpha=0.5, force=False):
    """Same as run_overlap_test_full but with snap-adaptive skip rule."""
    config_key = f'snap={snap_dbc:g}_alpha={alpha:g}_full_v2'
    cached = pd.read_pickle(OVERLAP_FULL_CACHE_PATH) if OVERLAP_FULL_CACHE_PATH.exists() else pd.DataFrame()
    if not force and not cached.empty and 'config' in cached.columns:
        if (cached['config'] == config_key).any():
            print(f'Loading cached results for {config_key}')
            return cached[cached['config'] == config_key].copy()

    print(f'Running {config_key}...')
    rows = []
    skip_log = {'no_obs': 0, 'low_first_review_dbc': 0, 'low_critics': 0}
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=snap_dbc)
        state = snapshot_state(target, snap_time)
        passed, reason = passes_skip_rules_for_snap(state, snap_dbc)
        if not passed:
            if reason and 'no obs' in reason:
                skip_log['no_obs'] += 1
            elif 'first_review_dbc' in (reason or ''):
                skip_log['low_first_review_dbc'] += 1
            else:
                skip_log['low_critics'] += 1
            continue

        target_window_days = state['first_review_dbc'] - snap_dbc
        target_critics = state['observed_critics']

        method_specs = [
            ('control_stratified', lambda: matched_training_slugs(target, target_gap, band=3.0, n=20)[0]),
            ('gap_overlap_ranked', lambda: gap_overlap_ranked_selector(
                target, target_gap, target_critics, target_window_days, k=20, gap_band=5.0,
            )[0]),
            ('combined_score', lambda: combined_score_selector(
                target, target_gap, target_critics, target_window_days, k=20, alpha=alpha, sigma_gap=8.0,
            )[0]),
        ]

        for method_name, training_fn in method_specs:
            training = training_fn()
            if len(training) < 5:
                rows.append({
                    'target_slug': target, 'target_gap': target_gap,
                    'snap_dbc': snap_dbc, 'method': method_name,
                    'predicted': np.nan, 'actual': np.nan,
                    'config': config_key,
                })
                continue
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            model = build_kde_lambda_model_capped(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)

            pred = predict_window(
                model, dbc_from=snap_dbc, dbc_to=0.0,
                observed_critics=state['observed_critics'],
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
            act = actual_remaining(target, snap_dbc)
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'snap_dbc': snap_dbc, 'method': method_name,
                'predicted': pred, 'actual': act,
                'config': config_key,
            })

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    print(f'  Skipped: {skip_log}')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df_combined = pd.concat([cached, df], ignore_index=True)
        df_combined = df_combined.drop_duplicates(['target_slug', 'snap_dbc', 'method', 'config'], keep='last')
    else:
        df_combined = df
    df_combined.to_pickle(OVERLAP_FULL_CACHE_PATH)
    return df_combined[df_combined['config'] == config_key].copy()

results_t5 = run_overlap_test_full_v2(snap_dbc=5.0, alpha=0.5)
print(f'\nTotal rows T-5d: {len(results_t5)}')


In [ ]:
summarize_full(results_t5, 'T-5d snap, predict (T-5d, midnight UTC of close]')


### 13.3 Alpha sweep on full window (combined_score only)

Sweep α ∈ {0.1, 0.3, 0.5, 0.7, 0.9} for combined_score at T-3d and T-5d full windows. Confirms whether α=0.5 is still optimal on deployment-relevant window.


In [ ]:
ALPHA_SWEEP_CACHE_PATH = CACHE_DIR / 'alpha_sweep_full.pkl'

def run_combined_score_only(snap_dbc, alpha, force=False):
    """Run combined_score only (skip control + gap_overlap_ranked) for fast alpha sweep."""
    config_key = f'snap={snap_dbc:g}_alpha={alpha:g}_combined_only'
    cached = pd.read_pickle(ALPHA_SWEEP_CACHE_PATH) if ALPHA_SWEEP_CACHE_PATH.exists() else pd.DataFrame()
    if not force and not cached.empty and 'config' in cached.columns:
        if (cached['config'] == config_key).any():
            return cached[cached['config'] == config_key].copy()

    print(f'Running {config_key}...')
    rows = []
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=snap_dbc)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, snap_dbc)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - snap_dbc
        target_critics = state['observed_critics']

        training = combined_score_selector(
            target, target_gap, target_critics, target_window_days,
            k=20, alpha=alpha, sigma_gap=8.0,
        )[0]
        if len(training) < 5:
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'snap_dbc': snap_dbc, 'alpha': alpha,
                'predicted': np.nan, 'actual': np.nan,
                'config': config_key,
            })
            continue

        profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
        model = build_kde_lambda_model_capped(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
        pred = predict_window(
            model, dbc_from=snap_dbc, dbc_to=0.0,
            observed_critics=state['observed_critics'],
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )
        act = actual_remaining(target, snap_dbc)
        rows.append({
            'target_slug': target, 'target_gap': target_gap,
            'snap_dbc': snap_dbc, 'alpha': alpha,
            'predicted': pred, 'actual': act,
            'config': config_key,
        })
        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{len(close_date_map)}')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df_combined = pd.concat([cached, df], ignore_index=True)
        df_combined = df_combined.drop_duplicates(['target_slug', 'snap_dbc', 'alpha', 'config'], keep='last')
    else:
        df_combined = df
    df_combined.to_pickle(ALPHA_SWEEP_CACHE_PATH)
    return df_combined[df_combined['config'] == config_key].copy()

# Sweep
sweep_rows = []
for snap in [3.0, 5.0]:
    for a in [0.1, 0.3, 0.5, 0.7, 0.9]:
        res = run_combined_score_only(snap, a)
        df = res.dropna(subset=['predicted', 'actual']).copy()
        df['abs_err'] = (df['predicted'] - df['actual']).abs()
        df['ratio'] = df['predicted'] / df['actual']
        clean = df[df['actual'] > 0]
        sweep_rows.append({
            'snap': f'T-{snap:g}d',
            'alpha': a,
            'n': len(df),
            'MAE': df['abs_err'].mean(),
            'median_ratio': clean['ratio'].median() if len(clean) else np.nan,
            'median_err': (df['predicted'] - df['actual']).median(),
        })

print('\n=== Alpha sweep, full window ===')
print(pd.DataFrame(sweep_rows).to_string(index=False, float_format='%.3f'))


### 13.4 Progression at T-5d snap (full window): how much better has the KDE gotten?

Take stock of cumulative improvement on the deployment-relevant T-5d snap (predict to midnight UTC of close, full window). Walk through 4 configurations:

1. **Original baseline:** `default_training_slugs` + Scott's rule (no cap)
2. **+ bandwidth cap:** same training + ceil=0.7d
3. **+ gap-stratified:** `matched_training_slugs(band=3)` + ceil=0.7d
4. **+ combined_score (current best):** `combined_score(α=0.5)` + ceil=0.7d

Compare on the same n_common subset. Same skip rules apply.


In [ ]:
PROGRESSION_CACHE_PATH = CACHE_DIR / 'progression_t5.pkl'

def run_progression_t5(force=False):
    config_key = 'progression_t5'
    cached = pd.read_pickle(PROGRESSION_CACHE_PATH) if PROGRESSION_CACHE_PATH.exists() else pd.DataFrame()
    if not force and not cached.empty and 'config' in cached.columns:
        if (cached['config'] == config_key).any():
            print(f'Loading cached results for {config_key}')
            return cached[cached['config'] == config_key].copy()

    rows = []
    snap_dbc = 5.0
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=snap_dbc)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, snap_dbc)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - snap_dbc
        target_critics = state['observed_critics']

        method_specs = [
            ('1_original_baseline', lambda: default_training_slugs(
                movies, exclude_slug=target, before_date=close_date_map[target], n=20,
            ), False),  # no bandwidth cap
            ('2_baseline_plus_cap', lambda: default_training_slugs(
                movies, exclude_slug=target, before_date=close_date_map[target], n=20,
            ), True),  # bandwidth cap
            ('3_gap_stratified_plus_cap', lambda: matched_training_slugs(
                target, target_gap, band=3.0, n=20,
            )[0], True),
            ('4_combined_score_plus_cap', lambda: combined_score_selector(
                target, target_gap, target_critics, target_window_days,
                k=20, alpha=0.5, sigma_gap=8.0,
            )[0], True),
        ]

        for method_name, training_fn, use_cap in method_specs:
            training = training_fn()
            if len(training) < 5:
                rows.append({
                    'target_slug': target, 'target_gap': target_gap,
                    'method': method_name,
                    'predicted': np.nan, 'actual': np.nan,
                    'config': config_key,
                })
                continue
            profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
            if use_cap:
                model = build_kde_lambda_model_capped(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
            else:
                model = build_kde_lambda_model(profiles, verbose=False)

            pred = predict_window(
                model, dbc_from=snap_dbc, dbc_to=0.0,
                observed_critics=state['observed_critics'],
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
            act = actual_remaining(target, snap_dbc)
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'method': method_name,
                'predicted': pred, 'actual': act,
                'config': config_key,
            })
        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{len(close_date_map)}')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df_combined = pd.concat([cached, df], ignore_index=True)
        df_combined = df_combined.drop_duplicates(['target_slug', 'method', 'config'], keep='last')
    else:
        df_combined = df
    df_combined.to_pickle(PROGRESSION_CACHE_PATH)
    return df_combined[df_combined['config'] == config_key].copy()

prog = run_progression_t5()
print(f'\nTotal rows: {len(prog)}')


In [ ]:
# Aggregate
prog_clean = prog.dropna(subset=['predicted', 'actual']).copy()
prog_clean['err'] = prog_clean['predicted'] - prog_clean['actual']
prog_clean['abs_err'] = prog_clean['err'].abs()
prog_clean['ratio'] = prog_clean['predicted'] / prog_clean['actual']

methods = ['1_original_baseline', '2_baseline_plus_cap', '3_gap_stratified_plus_cap', '4_combined_score_plus_cap']
common = prog_clean.groupby('target_slug')['method'].nunique() == len(methods)
prog_clean = prog_clean[prog_clean['target_slug'].isin(common[common].index)]

print(f'=== T-5d progression, full window | n_common = {prog_clean["target_slug"].nunique()} ===')
rows = []
baseline_mae = None
for m in methods:
    sub = prog_clean[prog_clean['method'] == m]
    clean_r = sub[sub['actual'] > 0]
    mae = sub['abs_err'].mean()
    if baseline_mae is None:
        baseline_mae = mae
    rel = (baseline_mae - mae) / baseline_mae * 100 if baseline_mae else 0
    rows.append({
        'method': m,
        'MAE': mae,
        'median_err': sub['err'].median(),
        'median_ratio': clean_r['ratio'].median() if len(clean_r) else np.nan,
        'rel_vs_orig': f'{rel:+.1f}%',
    })
print(pd.DataFrame(rows).to_string(index=False, float_format='%.3f'))

# Print incremental gains
print('\n--- Incremental MAE improvements ---')
prev_mae = None
for r in rows:
    if prev_mae is not None:
        delta = (prev_mae - r['MAE']) / prev_mae * 100
        print(f'  {r["method"]}: {delta:+.1f}% step over previous')
    prev_mae = r['MAE']


## 14. Phase B: shape similarity (early-arrival rate)

Adds early-arrival rate as a third feature on top of `combined_score`'s gap + Jaccard. For each (target, candidate), compute `early_rate = critics_in_window_count / target_window_days`. Similarity = `exp(-|rate_target - rate_candidate| / sigma_rate)` with `sigma_rate=2.0` (reviews/day scale).

New combined score with three features:
```
score = w_gap · exp(-|gap_diff|/8) + w_jaccard · jaccard + w_shape · exp(-|rate_diff|/2)
```

Test fixed equal weights (w=1/3 each), then sweep w_shape ∈ {0.1, 0.2, 0.3, 0.4, 0.5}. Compare to current best (combined_score, α=0.5, two features).

**Decision rule:** ≥3% MAE improvement at T-3d or T-5d full window → continue adding features (Phase C). Otherwise stop and ship validated stack.


In [ ]:
def early_rate(slug, window_start, window_days):
    """Reviews per day in the first window_days post window_start."""
    if window_days <= 0:
        return 0.0
    n_critics = len(critics_in_window(slug, window_start, window_days))
    return n_critics / window_days

def combined_3feature_selector(target, target_gap, target_critics, target_window_days,
                                target_rate, k=20, w_gap=1/3, w_jaccard=1/3, w_shape=1/3,
                                sigma_gap=8.0, sigma_rate=2.0):
    """Three-feature similarity ranking."""
    target_close = close_date_map[target]
    candidates = gaps[
        (gaps['close_ts'] < target_close)
        & (gaps['slug'] != target)
    ]
    if len(candidates) == 0:
        return [], 0.0
    rows = []
    for _, row in candidates.iterrows():
        slug = row['slug']
        gap_diff = abs(row['gap_days'] - target_gap)
        gap_score = float(np.exp(-gap_diff / sigma_gap))

        train_first = first_review_ts.loc[slug]
        train_critics = critics_in_window(slug, train_first, target_window_days)
        j = jaccard(target_critics, train_critics)

        train_rate = early_rate(slug, train_first, target_window_days)
        rate_diff = abs(target_rate - train_rate)
        shape_score = float(np.exp(-rate_diff / sigma_rate))

        combined = w_gap * gap_score + w_jaccard * j + w_shape * shape_score
        rows.append((slug, combined))
    rows.sort(key=lambda x: x[1], reverse=True)
    selected = [r[0] for r in rows[:k]]
    median_score = np.median([r[1] for r in rows[:k]]) if rows else 0.0
    return selected, float(median_score)

# Spot-check
sample_target = 'lilo_and_stitch_2025'
sample_gap = gap_for_slug(sample_target)
sample_close = close_date_map[sample_target]
sample_snap_time = sample_close - pd.Timedelta(days=3)
sample_state = snapshot_state(sample_target, sample_snap_time)
sample_first_dbc = sample_state['first_review_dbc']
sample_window_days = sample_first_dbc - 3.0
sample_rate = len(sample_state['observed_critics']) / sample_window_days
print(f'{sample_target}: target_rate = {sample_rate:.2f} reviews/day in window of {sample_window_days:.2f}d')

selected, score = combined_3feature_selector(
    sample_target, sample_gap, sample_state['observed_critics'], sample_window_days, sample_rate,
)
print(f'\n3-feature selector picked: {selected[:5]}')

# Compare rates of selected
for s in selected[:5]:
    s_first = first_review_ts.loc[s]
    s_rate = early_rate(s, s_first, sample_window_days)
    s_gap = gap_for_slug(s)
    print(f'  {s}: rate={s_rate:.2f}/d, gap={s_gap:.2f}d')


In [ ]:
PHASE_B_CACHE_PATH = CACHE_DIR / 'phase_b_shape.pkl'

def run_phase_b(snap_dbc, w_shape, force=False):
    config_key = f'snap={snap_dbc:g}_wshape={w_shape:g}'
    cached = pd.read_pickle(PHASE_B_CACHE_PATH) if PHASE_B_CACHE_PATH.exists() else pd.DataFrame()
    if not force and not cached.empty and 'config' in cached.columns:
        if (cached['config'] == config_key).any():
            return cached[cached['config'] == config_key].copy()

    print(f'Running {config_key}...')
    rows = []
    # With w_shape, the remaining 1-w_shape is split 50/50 between gap and jaccard (matches α=0.5)
    w_gap = (1 - w_shape) / 2
    w_jaccard = (1 - w_shape) / 2

    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=snap_dbc)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, snap_dbc)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - snap_dbc
        target_critics = state['observed_critics']
        target_rate = len(target_critics) / target_window_days if target_window_days > 0 else 0.0

        training = combined_3feature_selector(
            target, target_gap, target_critics, target_window_days, target_rate,
            k=20, w_gap=w_gap, w_jaccard=w_jaccard, w_shape=w_shape,
        )[0]
        if len(training) < 5:
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'snap_dbc': snap_dbc, 'w_shape': w_shape,
                'predicted': np.nan, 'actual': np.nan,
                'config': config_key,
            })
            continue
        profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
        model = build_kde_lambda_model_capped(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
        pred = predict_window(
            model, dbc_from=snap_dbc, dbc_to=0.0,
            observed_critics=state['observed_critics'],
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )
        act = actual_remaining(target, snap_dbc)
        rows.append({
            'target_slug': target, 'target_gap': target_gap,
            'snap_dbc': snap_dbc, 'w_shape': w_shape,
            'predicted': pred, 'actual': act,
            'config': config_key,
        })
        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{len(close_date_map)}')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df_combined = pd.concat([cached, df], ignore_index=True)
        df_combined = df_combined.drop_duplicates(['target_slug', 'snap_dbc', 'w_shape', 'config'], keep='last')
    else:
        df_combined = df
    df_combined.to_pickle(PHASE_B_CACHE_PATH)
    return df_combined[df_combined['config'] == config_key].copy()

# Sweep w_shape at T-3d and T-5d
sweep_b = []
for snap in [3.0, 5.0]:
    for w in [0.0, 0.1, 0.2, 0.33, 0.5]:  # 0.0 = baseline (combined_score), 0.33 = equal 3-feature
        res = run_phase_b(snap, w)
        df = res.dropna(subset=['predicted', 'actual']).copy()
        df['abs_err'] = (df['predicted'] - df['actual']).abs()
        df['ratio'] = df['predicted'] / df['actual']
        clean = df[df['actual'] > 0]
        sweep_b.append({
            'snap': f'T-{snap:g}d',
            'w_shape': w,
            'n': len(df),
            'MAE': df['abs_err'].mean(),
            'median_ratio': clean['ratio'].median() if len(clean) else np.nan,
            'median_err': (df['predicted'] - df['actual']).median(),
        })

print('\n=== Phase B w_shape sweep, full window ===')
print(pd.DataFrame(sweep_b).to_string(index=False, float_format='%.3f'))


## 15. Phase B (extended): recency feature

Tests whether recency of the candidate's close date relative to target's close date adds signal. Motivation: critic ecosystem may drift over time (new critics, retiring critics, changing publication policies). `combined_score` doesn't currently weight recency — picks based on gap + Jaccard regardless of when the candidate closed. Stratified training implicitly biases recent (n=20 most recent) but combined_score loses that bias.

`recency_score = exp(-|target_close - candidate_close| / sigma_recency)` with sigma=90d.

New combined: `w_gap · gap_score + w_jaccard · jaccard + w_recency · recency_score`. With w_recency varying, the remaining weight splits 50/50 between gap and jaccard (matches α=0.5 from combined_score).

**Decision rule (same as Phase B):** ≥3% MAE improvement at T-3d or T-5d full window → keep. Otherwise stop.


In [ ]:
def recency_score_fn(target, candidate, sigma_recency_days=90.0):
    target_close = close_date_map[target]
    cand_close = close_date_map[candidate]
    days_diff = abs((target_close - cand_close).total_seconds() / 86400)
    return float(np.exp(-days_diff / sigma_recency_days))

def combined_recency_selector(target, target_gap, target_critics, target_window_days,
                              k=20, w_gap=0.5, w_jaccard=0.5, w_recency=0.0,
                              sigma_gap=8.0, sigma_recency=90.0):
    """Three-feature similarity ranking with recency."""
    target_close = close_date_map[target]
    candidates = gaps[
        (gaps['close_ts'] < target_close)
        & (gaps['slug'] != target)
    ]
    if len(candidates) == 0:
        return [], 0.0
    rows = []
    for _, row in candidates.iterrows():
        slug = row['slug']
        gap_diff = abs(row['gap_days'] - target_gap)
        gap_score = float(np.exp(-gap_diff / sigma_gap))

        train_first = first_review_ts.loc[slug]
        train_critics = critics_in_window(slug, train_first, target_window_days)
        j = jaccard(target_critics, train_critics)

        rec = recency_score_fn(target, slug, sigma_recency_days=sigma_recency)

        combined = w_gap * gap_score + w_jaccard * j + w_recency * rec
        rows.append((slug, combined))
    rows.sort(key=lambda x: x[1], reverse=True)
    selected = [r[0] for r in rows[:k]]
    median_score = np.median([r[1] for r in rows[:k]]) if rows else 0.0
    return selected, float(median_score)

# Spot-check on a target
sample_target = 'lilo_and_stitch_2025'
sample_gap = gap_for_slug(sample_target)
sample_close = close_date_map[sample_target]
sample_snap_time = sample_close - pd.Timedelta(days=3)
sample_state = snapshot_state(sample_target, sample_snap_time)
sample_first_dbc = sample_state['first_review_dbc']
sample_window_days = sample_first_dbc - 3.0
print(f'Target: {sample_target}, close = {sample_close}')

# Without recency (combined_score baseline)
sel_no_rec, _ = combined_recency_selector(
    sample_target, sample_gap, sample_state['observed_critics'], sample_window_days,
    w_gap=0.5, w_jaccard=0.5, w_recency=0.0,
)
# With recency 0.3
sel_rec, _ = combined_recency_selector(
    sample_target, sample_gap, sample_state['observed_critics'], sample_window_days,
    w_gap=0.35, w_jaccard=0.35, w_recency=0.3,
)

# Show closes for picked movies
def avg_recency(slugs):
    diffs = [(sample_close - close_date_map[s]).total_seconds() / 86400 for s in slugs]
    return np.mean(diffs)

print(f'\nNo-recency picks: median {avg_recency(sel_no_rec):.0f}d before target close')
print(f'  first 5: {sel_no_rec[:5]}')
print(f'\nRecency=0.3 picks: median {avg_recency(sel_rec):.0f}d before target close')
print(f'  first 5: {sel_rec[:5]}')


In [ ]:
RECENCY_CACHE_PATH = CACHE_DIR / 'phase_b_recency.pkl'

def run_recency(snap_dbc, w_recency, force=False):
    config_key = f'snap={snap_dbc:g}_wrec={w_recency:g}'
    cached = pd.read_pickle(RECENCY_CACHE_PATH) if RECENCY_CACHE_PATH.exists() else pd.DataFrame()
    if not force and not cached.empty and 'config' in cached.columns:
        if (cached['config'] == config_key).any():
            return cached[cached['config'] == config_key].copy()

    print(f'Running {config_key}...')
    rows = []
    w_gap = (1 - w_recency) / 2
    w_jaccard = (1 - w_recency) / 2

    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        snap_time = target_close - pd.Timedelta(days=snap_dbc)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, snap_dbc)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - snap_dbc
        target_critics = state['observed_critics']

        training = combined_recency_selector(
            target, target_gap, target_critics, target_window_days,
            k=20, w_gap=w_gap, w_jaccard=w_jaccard, w_recency=w_recency,
        )[0]
        if len(training) < 5:
            rows.append({
                'target_slug': target, 'target_gap': target_gap,
                'snap_dbc': snap_dbc, 'w_recency': w_recency,
                'predicted': np.nan, 'actual': np.nan,
                'config': config_key,
            })
            continue
        profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
        model = build_kde_lambda_model_capped(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
        pred = predict_window(
            model, dbc_from=snap_dbc, dbc_to=0.0,
            observed_critics=state['observed_critics'],
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )
        act = actual_remaining(target, snap_dbc)
        rows.append({
            'target_slug': target, 'target_gap': target_gap,
            'snap_dbc': snap_dbc, 'w_recency': w_recency,
            'predicted': pred, 'actual': act,
            'config': config_key,
        })
        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{len(close_date_map)}')

    df = pd.DataFrame(rows)
    if not cached.empty:
        df_combined = pd.concat([cached, df], ignore_index=True)
        df_combined = df_combined.drop_duplicates(['target_slug', 'snap_dbc', 'w_recency', 'config'], keep='last')
    else:
        df_combined = df
    df_combined.to_pickle(RECENCY_CACHE_PATH)
    return df_combined[df_combined['config'] == config_key].copy()

sweep_rec = []
for snap in [3.0, 5.0]:
    for w in [0.0, 0.1, 0.2, 0.33, 0.5]:
        res = run_recency(snap, w)
        df = res.dropna(subset=['predicted', 'actual']).copy()
        df['abs_err'] = (df['predicted'] - df['actual']).abs()
        df['ratio'] = df['predicted'] / df['actual']
        clean = df[df['actual'] > 0]
        sweep_rec.append({
            'snap': f'T-{snap:g}d',
            'w_recency': w,
            'n': len(df),
            'MAE': df['abs_err'].mean(),
            'median_ratio': clean['ratio'].median() if len(clean) else np.nan,
            'median_err': (df['predicted'] - df['actual']).median(),
        })

print('\n=== Recency w_recency sweep, full window ===')
print(pd.DataFrame(sweep_rec).to_string(index=False, float_format='%.3f'))
